<h1 style="text-align: center;">Data Understanding</h1>
<h3 style="text-align: center;">Hotel Booking Cancellation Prediction</h3>

---

<h5 style="text-align: right;">By Beta Group</h5>

# **Section 0: Setup**

## **0.1. Import Library**

In [1]:
import numpy as np
import pandas as pd

## **0.2. Global Configuration**

In [2]:
import warnings
warnings.filterwarnings("ignore")

RANDOM_STATE=42

## **0.3. Load Dataset**

In [3]:
df_raw = pd.read_csv("../data/hotel_bookings_2017.csv")
df_raw.head()

,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date
0,Resort Hotel,0,342,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
1,Resort Hotel,0,737,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
2,Resort Hotel,0,7,2015,July,27,1,0,1,1,...,No Deposit,NaN,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02
3,Resort Hotel,0,13,2015,July,27,1,0,1,1,...,No Deposit,304.0,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02
4,Resort Hotel,0,14,2015,July,27,1,0,2,2,...,No Deposit,240.0,NaN,0,Transient,98.0,0,1,Check-Out,2015-07-03


# **Section 1: Data Explanation**

**Dataset Overview**


- **Source File:** `hotel_bookings_2017.csv`
- **Initial Dataset Size:** 119,390 catatan booking
- **Initial Feature Dimensionality:** 32 kolom
- **Unit of Analysis:** satu baris mewakili satu catatan booking hotel
- **Target Variable:** `is_canceled`
- **Target Definition:** `1` = canceled and `0` = not canceled
- **Problem Type:** Supervised Binary Classification
- **Prediction Goal:** Memprediksi kemungkinan sebuah booking hotel akan dibatalkan


## **1.1. General Information**

In [4]:
df_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 119390 entries, 0 to 119389
Data columns (total 32 columns):
 #   Column                          Non-Null Count   Dtype  
---  ------                          --------------   -----  
 0   hotel                           119390 non-null  object 
 1   is_canceled                     119390 non-null  int64  
 2   lead_time                       119390 non-null  int64  
 3   arrival_date_year               119390 non-null  int64  
 4   arrival_date_month              119390 non-null  object 
 5   arrival_date_week_number        119390 non-null  int64  
 6   arrival_date_day_of_month       119390 non-null  int64  
 7   stays_in_weekend_nights         119390 non-null  int64  
 8   stays_in_week_nights            119390 non-null  int64  
 9   adults                          119390 non-null  int64  
 10  children                        119386 non-null  float64
 11  babies                          119390 non-null  int64  
 12  meal            

In [5]:
df_raw.columns

Index(['hotel', 'is_canceled', 'lead_time', 'arrival_date_year',
       'arrival_date_month', 'arrival_date_week_number',
       'arrival_date_day_of_month', 'stays_in_weekend_nights',
       'stays_in_week_nights', 'adults', 'children', 'babies', 'meal',
       'country', 'market_segment', 'distribution_channel',
       'is_repeated_guest', 'previous_cancellations',
       'previous_bookings_not_canceled', 'reserved_room_type',
       'assigned_room_type', 'booking_changes', 'deposit_type', 'agent',
       'company', 'days_in_waiting_list', 'customer_type', 'adr',
       'required_car_parking_spaces', 'total_of_special_requests',
       'reservation_status', 'reservation_status_date'],
      dtype='object')

In [6]:
df_raw.head()

,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date
0,Resort Hotel,0,342,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
1,Resort Hotel,0,737,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
2,Resort Hotel,0,7,2015,July,27,1,0,1,1,...,No Deposit,NaN,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02
3,Resort Hotel,0,13,2015,July,27,1,0,1,1,...,No Deposit,304.0,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02
4,Resort Hotel,0,14,2015,July,27,1,0,2,2,...,No Deposit,240.0,NaN,0,Transient,98.0,0,1,Check-Out,2015-07-03


### **Temuan Utama**

- Dataset awal terdiri dari **119.390 baris dan 32 kolom**.
- Tipe data awal mencakup:

    * **16 kolom bertipe integer**
    * **4 kolom  bertipe float** 
    * **12 kolom bertipe object**

Beberapa permasalahan awal yang ditemukan adalah:

- `reservation_status_date` masih bertipe object, meskipun isinya merepresentasikan tanggal.
- `agent` dan `company` bertipe numerik, padahal nilainya merupakan kode identitas.
- `children` → Ubah ke `int64`
- `agent` → Ubah ke `int64`
- `company` → Ubah ke `int64`


### **Insight**

Terdapat ketidaksesuaian tipe data dan potensi kebocoran informasi.

Kode `agent` dan `company` tidak memiliki makna kuantitatif. Model tidak boleh menafsirkan bahwa agent 240 lebih besar atau lebih penting daripada agent 9 hanya karena nilai angkanya lebih tinggi. Oleh karena itu, kedua fitur tersebut perlu diperlakukan sebagai kategori.

## **1.2. Feature Understanding**


**Feature Variables**
| Feature                          | Data Type                | Description                                                                                                                                                 |
| -------------------------------- | ------------------------ | ----------------------------------------------------------------------------------------------------------------------------------------------------------- |
| `hotel`                          | Categorical              | Type of hotel where the booking was made, either **Resort Hotel** or **City Hotel**.                                                                        |
| `lead_time`                      | Numerical                | Number of days between the date when the booking was entered into the PMS and the guest's arrival date.                                                     |
| `arrival_date_year`              | Numerical / Temporal     | Year in which the guest is scheduled to arrive.                                                                                                             |
| `arrival_date_month`             | Categorical / Temporal   | Month in which the guest is scheduled to arrive.                                                                                                            |
| `arrival_date_week_number`       | Numerical / Temporal     | Week number of the year in which the guest is scheduled to arrive.                                                                                          |
| `arrival_date_day_of_month`      | Numerical / Temporal     | Day of the month on which the guest is scheduled to arrive.                                                                                                 |
| `stays_in_weekend_nights`        | Numerical                | Number of weekend nights (Saturday or Sunday) the guest stays or books.                                                                                     |
| `stays_in_week_nights`           | Numerical                | Number of week nights (Monday to Friday) the guest stays or books.                                                                                          |
| `adults`                         | Numerical                | Number of adults included in the booking.                                                                                                                   |
| `children`                       | Numerical                | Number of children included in the booking.                                                                                                                 |
| `babies`                         | Numerical                | Number of babies included in the booking.                                                                                                                   |
| `meal`                           | Categorical              | Type of meal plan included in the booking. `BB` = Bed & Breakfast, `FB` = Full Board, `HB` = Half Board, `SC` = Self Catering, `Undefined` = not specified. |
| `country`                        | Categorical              | Country of origin of the guest.                                                                                                                             |
| `market_segment`                 | Categorical              | Market segment through which the booking was made, such as Direct, Corporate, Online TA, Offline TA/TO, etc.                                                |
| `distribution_channel`           | Categorical              | Distribution channel through which the booking was received, such as Direct, Corporate, TA/TO, or GDS.                                                      |
| `is_repeated_guest`              | Binary                   | Indicates whether the guest has previously stayed at the hotel. `1` = repeated guest, `0` = new guest.                                                      |
| `previous_cancellations`         | Numerical                | Number of previous bookings that were canceled by the guest before the current booking.                                                                     |
| `previous_bookings_not_canceled` | Numerical                | Number of previous bookings that were not canceled by the guest before the current booking.                                                                 |
| `reserved_room_type`             | Categorical              | Room type that the guest originally reserved.                                                                                                               |
| `assigned_room_type`             | Categorical              | Room type that was ultimately assigned to the guest.                                                                                                        |
| `booking_changes`                | Numerical                | Number of changes made to the booking after the original booking was created.                                                                               |
| `deposit_type`                   | Categorical              | Type of deposit/payment arrangement associated with the booking, such as No Deposit, Non Refund, or Refundable.                                             |
| `agent`                          | Categorical / Identifier | ID of the travel agent that made the booking. `NULL` indicates that no agent was involved or recorded.                                                      |
| `company`                        | Categorical / Identifier | ID of the company/entity responsible for the booking. `NULL` indicates that no company was associated with the booking.                                     |
| `days_in_waiting_list`           | Numerical                | Number of days the booking remained on the waiting list before being confirmed.                                                                             |
| `customer_type`                  | Categorical              | Type of customer associated with the booking, such as Transient, Contract, Transient-Party, or Group.                                                       |
| `adr`                            | Numerical                | **Average Daily Rate** — average revenue generated per occupied room per day.                                                                               |
| `required_car_parking_spaces`    | Numerical                | Number of car parking spaces required by the guest.                                                                                                         |
| `total_of_special_requests`      | Numerical                | Total number of special requests made by the guest, such as room preferences or other arrangements.                                                         |
| `reservation_status`             | Categorical              | Final status of the reservation, such as Canceled, Check-Out, or No-Show.                                                                                   |
| `reservation_status_date`        | Date / Temporal          | Date on which the latest reservation status was recorded.                                                                                                   |



**Target Variable**
| Target                          | Data Type                | Description                                                                                                                                                 |
| -------------------------------- | ------------------------ | -----------------------------------------------------------------------------------------------------------------------------------------------------------
| `is_canceled`                    | Binary                   | Indicates whether the booking was canceled. `1` = canceled, `0` = not canceled. **Target variable.**                                                        |


## **1.3. Data Type Classification**

In [7]:
num_features = [
    "lead_time",
    "arrival_date_week_number",
    "arrival_date_day_of_month",
    "stays_in_weekend_nights",
    "stays_in_week_nights",
    "adults",
    "children",
    "babies",
    "previous_cancellations",
    "previous_bookings_not_canceled",
    "booking_changes",
    "days_in_waiting_list",
    "adr",
    "required_car_parking_spaces",
    "total_of_special_requests",
]

cat_features = [
    "hotel",
    "arrival_date_year",
    "arrival_date_month",
    "meal",
    "country",
    "market_segment",
    "distribution_channel",
    "reserved_room_type",
    "assigned_room_type",
    "deposit_type",
    "customer_type",
    "agent",
    "company"
]

## **1.4. Statistic Summary**

In [8]:
df_raw.describe(include="all")

,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date
count,119390,119390.000000,119390.000000,119390.000000,119390,119390.000000,119390.000000,119390.000000,119390.000000,119390.000000,...,119390,103050.000000,6797.000000,119390.000000,119390,119390.000000,119390.000000,119390.000000,119390,119390
unique,2,NaN,NaN,NaN,12,NaN,NaN,NaN,NaN,NaN,...,3,NaN,NaN,NaN,4,NaN,NaN,NaN,3,926
top,City Hotel,NaN,NaN,NaN,August,NaN,NaN,NaN,NaN,NaN,...,No Deposit,NaN,NaN,NaN,Transient,NaN,NaN,NaN,Check-Out,2015-10-21
freq,79330,NaN,NaN,NaN,13877,NaN,NaN,NaN,NaN,NaN,...,104641,NaN,NaN,NaN,89613,NaN,NaN,NaN,75166,1461
mean,NaN,0.370416,104.011416,2016.156554,NaN,27.165173,15.798241,0.927599,2.500302,1.856403,...,NaN,86.693382,189.266735,2.321149,NaN,101.831122,0.062518,0.571363,NaN,NaN
std,NaN,0.482918,106.863097,0.707476,NaN,13.605138,8.780829,0.998613,1.908286,0.579261,...,NaN,110.774548,131.655015,17.594721,NaN,50.535790,0.245291,0.792798,NaN,NaN
min,NaN,0.000000,0.000000,2015.000000,NaN,1.000000,1.000000,0.000000,0.000000,0.000000,...,NaN,1.000000,6.000000,0.000000,NaN,-6.380000,0.000000,0.000000,NaN,NaN
25%,NaN,0.000000,18.000000,2016.000000,NaN,16.000000,8.000000,0.000000,1.000000,2.000000,...,NaN,9.000000,62.000000,0.000000,NaN,69.290000,0.000000,0.000000,NaN,NaN
50%,NaN,0.000000,69.000000,2016.000000,NaN,28.000000,16.000000,1.000000,2.000000,2.000000,...,NaN,14.000000,179.000000,0.000000,NaN,94.575000,0.000000,0.000000,NaN,NaN
75%,NaN,1.000000,160.000000,2017.000000,NaN,38.000000,23.000000,2.000000,3.000000,2.000000,...,NaN,229.000000,270.000000,0.000000,NaN,126.000000,0.000000,1.000000,NaN,NaN


### **Temuan Utama**

- Rata-rata `is_canceled` sebesar **0,3704**, yang menunjukkan bahwa sekitar **37,04% reservasi dibatalkan**.
- Median `lead_time` adalah **69 hari**, sedangkan rata-ratanya **104 hari**.
- Reservasi tipikal mencatat **2 orang dewasa**.
- Median durasi menginap adalah **1 malam akhir pekan dan 2 malam hari kerja**.
- Median ADR adalah **94,58**, sedangkan rata-ratanya **101,83**.
- Ditemukan ADR minimum **−6,38** dan maksimum **5.400**.
- Nilai maksimum `adult` mencapai 55, sedangkan `children` dan `babies` masing-masing mencapai **10**.

### **Insight**

Rata-rata `lead_time` yang jauh lebih tinggi daripada median menunjukkan distribusi menceng ke kanan, mayoritas reservasi dibuat dalam jarak waktu yang lebih pendek, tetapi sebagian kecil dibuat sangat jauh sebelum kedatangan.

Pada ADR dan komposisi tamu, nilai minimum atau maksimum yang ekstrem belum otomatis merupakan kesalahan. Nilai tersebut perlu dibandingkan dengan tipe kamar, segmen pasar, jumlah tamu, dan karakteristik reservasi lainnya sebelum keputusan cleaning dibuat.




## **1.5. Target Variable Analysis**

In [9]:
df_raw["is_canceled"].value_counts()

is_canceled
0    75166
1    44224
Name: count, dtype: int64

In [10]:
df_raw["is_canceled"].value_counts(normalize=True) * 100

is_canceled
0    62.958372
1    37.041628
Name: proportion, dtype: float64

### **Penjelasan**

`is_canceled` merupakan target yang akan diprediksi oleh model:

- `0` menunjukkan reservasi tidak dibatalkan.
- `1` menunjukkan reservasi dibatalkan.

Distribusi target diperiksa untuk mengetahui keseimbangan kelas sebelum proses modelling.

### **Temuan**

| Booking Status | Total Records | Proportion |
|---|---:|---:|
| Not Canceled | 75,166 | 62.96% |
| Canceled | 44,224 | 37.04% |

### **Insight**

Lebih dari **1 dari setiap 3 reservasi berakhir dengan pembatalan**, sehingga permasalahan ini cukup material untuk dianalisis.

Distribusi kelas menunjukkan ketidakseimbangan moderat, tetapi belum tergolong ekstrem. Meskipun demikian, accuracy tidak cukup digunakan sebagai satu-satunya ukuran keberhasilan model karena model yang lebih sering memilih kelas mayoritas dapat menghasilkan accuracy yang terlihat baik tanpa cukup mampu mendeteksi pembatalan.

Evaluasi model perlu melibatkan ROC-AUC, precision, recall, F1-score, confusion matrix, dan penyesuaian threshold berdasarkan konsekuensi bisnis dari kesalahan prediksi.

# **Section 2: Data Quality Assessment**

## **2.1. Missing Value Assessment**

### 2.1.1 Explicit Missing Values

In [11]:
print(' === MISSING VALUES ===')
missing_summary = pd.DataFrame({
    'Missing Count': df_raw.isnull().sum(),
    'Missing (%)': (df_raw.isnull().sum() / len(df_raw) * 100).round(2)
})
print(f'Total missing values: {missing_summary['Missing Count'].sum()}')
display(missing_summary[missing_summary['Missing Count'] > 0].sort_values('Missing (%)', ascending=False))

 === MISSING VALUES ===
Total missing values: 129425


,Missing Count,Missing (%)
company,112593,94.31
agent,16340,13.69
country,488,0.41
children,4,0.00


| Fitur | Jumlah Missing | Persentase |
|---|---:|---:|
| `company` | 112.593 | 94,31% |
| `agent` | 16.340 | 13,69% |
| `country` | 488 | 0,41% |
| `children` | 4 | ±0,003% |

Missing value pada `company` sangat tinggi, sedangkan missing value pada `children` dan `country` relatif kecil.

### **Insight**

Besarnya missing value tidak dapat menjadi satu-satunya dasar untuk menghapus fitur. Pada `agent` dan `company`, nilai kosong kemungkinan memiliki arti operasional, yaitu reservasi tidak terhubung dengan agen atau perusahaan.

Karena itu, mengubahnya menjadi kategori `no agent` dan `no company` lebih informatif daripada menghapus kolom atau barisnya. Namun, interpretasi ini tetap harus dicatat sebagai asumsi karena tidak dapat dibuktikan sepenuhnya dari dataset.

### 2.1.2 Country Missing Value Pattern

In [ ]:
country_missing = df_raw[df_raw["country"].isna()].copy()

print("=== COUNTRY MISSING VALUE SUMMARY ===")
print(f"Total country missing: {len(country_missing):,}")
print(
    f"Percentage: "
    f"{len(country_missing) / len(df_raw) * 100:.2f}%"
)

# Perbandingan berdasarkan distribution channel
channel_comparison = pd.concat(
    [
        country_missing["distribution_channel"]
        .value_counts(normalize=True)
        .mul(100)
        .round(2)
        .rename("Country Missing (%)"),

        df_raw["distribution_channel"]
        .value_counts(normalize=True)
        .mul(100)
        .round(2)
        .rename("All Data (%)")
    ],
    axis=1
).fillna(0)

print("\n=== DISTRIBUTION CHANNEL COMPARISON ===")
display(channel_comparison)

# Perbandingan berdasarkan market segment
segment_comparison = pd.concat(
    [
        country_missing["market_segment"]
        .value_counts(normalize=True)
        .mul(100)
        .round(2)
        .rename("Country Missing (%)"),

        df_raw["market_segment"]
        .value_counts(normalize=True)
        .mul(100)
        .round(2)
        .rename("All Data (%)")
    ],
    axis=1
).fillna(0)

print("\n=== MARKET SEGMENT COMPARISON ===")
display(segment_comparison)

=== COUNTRY MISSING VALUE SUMMARY ===
Total country missing: 488
Percentage: 0.41%

=== DISTRIBUTION CHANNEL COMPARISON ===


,Country Missing (%),All Data (%)
distribution_channel,,
Corporate,38.11,5.59
Direct,33.20,12.27
TA/TO,28.69,81.98
GDS,0.00,0.16
Undefined,0.00,0.00



=== MARKET SEGMENT COMPARISON ===


,Country Missing (%),All Data (%)
market_segment,,
Corporate,37.70,4.44
Direct,32.17,10.56
Online TA,15.16,47.30
Offline TA/TO,12.09,20.29
Complementary,1.84,0.62
Groups,1.02,16.59
Aviation,0.00,0.20
Undefined,0.00,0.00


### **Insight**

Missing value pada `country` tidak tersebar secara merata. Reservasi dengan negara yang tidak diketahui memiliki proporsi saluran Corporate dan Direct yang jauh lebih tinggi dibandingkan keseluruhan dataset.

Temuan ini menunjukkan bahwa missing value `country` kemungkinan berkaitan dengan proses pencatatan pada saluran tertentu. Oleh karena itu, menghapus seluruh baris dengan `country` kosong dapat mengurangi representasi reservasi Corporate dan Direct.

Nilai tersebut lebih aman dipertahankan sebagai kategori `Unknown`.

## **2.2. Duplicate Records Assessment**

In [13]:
duplicate_count = df_raw.duplicated().sum()
duplicate_percentage = duplicate_count / len(df_raw) * 100

print(f"Total duplicated records: {duplicate_count:,}")
print(f"Duplicated percentage: {duplicate_percentage:.2f}%")

Total duplicated records: 31,994
Duplicated percentage: 26.80%


### **Penjelasan**

Pemeriksaan menemukan **31.994 baris identik**, setara dengan sekitar **26,80% dari keseluruhan dataset**.

Pandas menandai sebuah baris sebagai duplikat apabila seluruh nilai kolomnya sama dengan baris lain.

### **Insight**

Proporsi baris identik yang mencapai lebih dari seperempat dataset merupakan isu yang material. Jika baris tersebut benar-benar merupakan penggandaan data, pola tertentu dapat memperoleh bobot berlebihan dan performa model berpotensi terlihat lebih baik daripada kondisi sebenarnya.

Namun, dataset tidak memiliki ID reservasi unik. Oleh karena itu, dua reservasi berbeda masih mungkin mempunyai karakteristik yang sama. Baris tersebut sebaiknya tidak langsung dihapus sebelum sumber dan proses pembentukan dataset dapat dipastikan.

## **2.3. Categorical Frequency Analysis**

In [14]:
high_cardinality_features = ["country", "agent", "company"]

for feature in cat_features:

    frequency = df_raw[feature].value_counts(dropna=False)
    percentage = (
        df_raw[feature]
        .value_counts(normalize=True, dropna=False)
        .mul(100)
        .round(2)
    )

    summary = pd.DataFrame({
        "Frequency": frequency,
        "Percentage (%)": percentage
    })

    print(f"\n=== {feature.upper()} ===")
    print(f"Total unique categories: {df_raw[feature].nunique(dropna=False)}")

    if feature in high_cardinality_features:
        print("Showing the 10 most frequent categories:")
        display(summary.head(10))
    else:
        display(summary)


=== HOTEL ===
Total unique categories: 2


,Frequency,Percentage (%)
hotel,,
City Hotel,79330,66.45
Resort Hotel,40060,33.55



=== ARRIVAL_DATE_YEAR ===
Total unique categories: 3


,Frequency,Percentage (%)
arrival_date_year,,
2016,56707,47.50
2017,40687,34.08
2015,21996,18.42



=== ARRIVAL_DATE_MONTH ===
Total unique categories: 12


...

,Frequency,Percentage (%)
customer_type,,
Transient,89613,75.06
Transient-Party,25124,21.04
Contract,4076,3.41
Group,577,0.48



=== AGENT ===
Total unique categories: 334
Showing the 10 most frequent categories:


,Frequency,Percentage (%)
agent,,
9.0,31961,26.77
NaN,16340,13.69
240.0,13922,11.66
1.0,7191,6.02
14.0,3640,3.05
7.0,3539,2.96
6.0,3290,2.76
250.0,2870,2.40
241.0,1721,1.44



=== COMPANY ===
Total unique categories: 353
Showing the 10 most frequent categories:


,Frequency,Percentage (%)
company,,
NaN,112593,94.31
40.0,927,0.78
223.0,784,0.66
67.0,267,0.22
45.0,250,0.21
153.0,215,0.18
174.0,149,0.12
219.0,141,0.12
281.0,138,0.12


### **Penjelasan**

Frekuensi dan persentase setiap kategori diperiksa untuk mengetahui kategori dominan, kategori langka, serta potensi ketidakseimbangan representasi.

### **Temuan dan Insight**

#### **Hotel Type**

City Hotel mencakup **66,45%** data, sedangkan Resort Hotel mencakup **33,55%**.

Dataset lebih banyak merepresentasikan City Hotel. Oleh karena itu, performa model perlu diperiksa secara terpisah untuk kedua jenis hotel agar performa agregat tidak menutupi kelemahan pada Resort Hotel.

#### **Arrival Year**

Tahun 2016 mencakup **47,50%** data dan menjadi periode paling dominan.

Komposisi data yang berbeda antartahun menunjukkan pentingnya validasi berbasis waktu. Pengujian pada periode yang lebih baru dapat memberikan gambaran yang lebih realistis mengenai kemampuan model menghadapi reservasi masa depan.

#### **Arrival Month**

Agustus mempunyai volume reservasi terbesar sebesar **11,62%**, sedangkan Januari merupakan yang terendah sebesar **4,97%**.

Hal ini menunjukkan adanya pola musiman pada volume reservasi. Namun, volume tinggi belum berarti cancellation rate tinggi; hubungan bulan kedatangan dengan target perlu dianalisis secara terpisah.

#### **Meal**

Sebanyak **77,32%** reservasi menggunakan paket BB. Kategori `Undefined` mencakup **0,98%** data.

Dominasi BB menunjukkan rendahnya variasi pada pilihan paket makan. Kategori `Undefined` tidak memiliki makna bisnis yang pasti sehingga perlu distandardisasi menjadi `Unknown`.

#### **Country**

Portugal menjadi negara asal paling dominan dengan **40,87%**, diikuti Inggris **10,20%** dan Prancis **8,76%**.

Data sangat terkonsentrasi pada beberapa negara Eropa. Kinerja model perlu dipantau pada negara dengan observasi sedikit karena estimasi risikonya cenderung kurang stabil.

#### **Market Segment**

Online TA mencakup **47,30%** seluruh reservasi, diikuti Offline TA/TO **20,29%** dan Groups **16,59%**.

Online TA merupakan segmen dengan eksposur operasional terbesar. Namun, kontribusi terhadap total pembatalan harus dinilai menggunakan kombinasi volume reservasi dan cancellation rate, bukan volume saja.

#### **Distribution Channel**

TA/TO mendominasi dengan **81,98%**, sedangkan Direct hanya **12,27%**.

Hotel sangat bergantung pada jalur TA/TO. Perlu dibedakan bahwa `market_segment` menjelaskan kelompok pasar, sedangkan `distribution_channel` menjelaskan jalur reservasi masuk.

#### **Room Type**

Room type A mencakup **72,03% kamar yang dipesan**, tetapi hanya **62,03% kamar yang diberikan**.

Perbedaan tersebut menunjukkan adanya perpindahan tipe kamar antara tahap pemesanan dan penempatan. Akan tetapi, analisis tambahan diperlukan untuk mengetahui apakah perpindahan tersebut merupakan upgrade, downgrade, atau kebutuhan operasional.


#### **Deposit Type**

Sebanyak **87,65%** reservasi tidak menggunakan deposit, sedangkan Non Refund mencakup **12,22%**.

Kebijakan tanpa deposit sangat dominan, tetapi distribusi ini belum membuktikan bahwa tidak adanya deposit menyebabkan pembatalan. Perbandingan cancellation rate per deposit type diperlukan sebelum membuat rekomendasi kebijakan.

#### **Customer Type**

Transient mencakup **75,06%** reservasi.

Karena tiga dari empat reservasi berasal dari pelanggan Transient, performa model secara keseluruhan kemungkinan besar didominasi oleh pola pelanggan tersebut. Evaluasi per customer type diperlukan untuk menilai konsistensi model.

#### **Agent and Company**

Beberapa kode agent dan company muncul jauh lebih sering daripada kode lainnya.

Kode tersebut merupakan identitas, bukan ukuran numerik. Jika digunakan secara langsung sebagai angka, model dapat menciptakan hubungan urutan yang tidak mempunyai arti bisnis. Keduanya perlu diubah menjadi kategori atau diringkas menjadi indikator penggunaan agen/perusahaan.

## **2.4. Potential Data Leakage Assessment**

In [15]:
confirmed_leakage_features = [
    "reservation_status",
    "reservation_status_date"
]

### **Penjelasan**

Data leakage terjadi ketika model menggunakan informasi yang belum tersedia pada saat prediksi dilakukan atau secara langsung mengungkap hasil target.

Pada dataset ini:

- `reservation_status` berisi status akhir seperti Canceled, Check-Out, atau No-Show.
- `reservation_status_date` mencatat tanggal ketika status akhir ditentukan.

Kedua fitur tersebut terbentuk setelah perjalanan reservasi berlangsung dan berkaitan langsung dengan target `is_canceled`.

### **Insight**

Apabila kedua kolom digunakan dalam model, skor evaluasi dapat terlihat sangat tinggi karena model tidak benar-benar memprediksi pembatalan, tetapi membaca hasil akhir reservasi.

Oleh karena itu, `reservation_status` dan `reservation_status_date` harus dihapus sebelum modelling agar evaluasi mencerminkan kondisi penggunaan nyata.

# **Section 3: Anomaly and Outlier Investigation**

## **3.1. Numerical Outlier Detection**

In [16]:
for outlier in num_features: 
    q1 = df_raw[outlier].quantile(0.25)
    q3 = df_raw[outlier].quantile(0.75)
    iqr = q3 - q1

    upper = q3 + 1.5 * iqr
    lower = q1 - 1.5 * iqr

    anomalies = (df_raw[
                    (df_raw[outlier] < lower) | 
                    (df_raw[outlier] > upper) |
                    (df_raw[outlier] < 0)
                ][outlier].count())

    anomaly_values = (df_raw[
                        (df_raw[outlier] < lower) | 
                        (df_raw[outlier] > upper) |
                        (df_raw[outlier] < 0)
                    ][outlier].unique())

    max_values = df_raw[outlier].max()
    min_values = df_raw[outlier].min()

    print(f'OUTLIER FITUR {outlier}')
    print(f'Batas bawah    : {lower}')
    print(f'Batas atas     : {upper}')
    print(f'Jumlah anomali : {anomalies}')
    print(f'Nilai anomali  : {anomaly_values}')
    print(f'Nilai maximum  : {max_values}')
    print(f'Nilai minimum  : {min_values}')
    print()

OUTLIER FITUR lead_time
Batas bawah    : -195.0
Batas atas     : 373.0
Jumlah anomali : 3005
Nilai anomali  : [737 394 460 381 382 709 468 398 424 434 374 406 400 379 399 385 422 390
 376 375 397 542 403 383 384 393 435 386 378 471 462 411 450 454 532 445
 389 388 407 443 437 451 391 405 412 419 420 426 433 440 429 418 447 461
 605 457 475 464 482 626 489 496 503 510 517 524 531 538 545 552 559 566
 573 580 587 594 601 608 615 622 629 396 410 395 423 408 409 448 465 387
 414 476 479 467 490 493 478 504 507 458 518 521 377 444 380 463]
Nilai maximum  : 737
Nilai minimum  : 0

OUTLIER FITUR arrival_date_week_number
Batas bawah    : -17.0
Batas atas     : 71.0
Jumlah anomali : 0
Nilai anomali  : []
Nilai maximum  : 53
Nilai minimum  : 1

OUTLIER FITUR arrival_date_day_of_month
Batas bawah    : -14.5
Batas atas     : 45.5
Jumlah anomali : 0
Nilai anomali  : []
Nilai maximum  : 31
Nilai minimum  : 1

OUTLIER FITUR stays_in_weekend_nights
Batas bawah    : -3.0
Batas atas     : 5.0
Jumlah ano

### **Temuan Utama**

- `lead_time`: 3.005 outlier.
- `stays_in_weekend_nights`: 265 outlier.
- `stays_in_week_nights`: 3.354 outlier.
- `adults`: 29.710 outlier.
- `children`: 8.590 outlier.
- `babies`: 917 outlier.
- `previous_cancellations` : 6484 outlier.
- `previous_bookings_not_canceled` : 3620 outlier.
- `booking_changes` : 18076 outlier.
- `days_in_waiting_list`: 3.698 outlier.
- `adr`: 3.794 outlier.
- `required_car_parking_spaces` : 7416 outlier.
- `total_of_special_requests` : 2877 outlier.

### **Insight**

IQR menghasilkan batas nol pada fitur seperti `children`, `babies`, `previous_cancellations`, dan `days_in_waiting_list` karena setidaknya 75% nilainya adalah nol. Akibatnya, setiap nilai di atas nol ditandai sebagai outlier meskipun secara bisnis valid.

Hal ini menunjukkan bahwa deteksi statistik hanya berfungsi sebagai tahap penyaringan awal. Keputusan cleaning harus didasarkan pada kombinasi frekuensi, konteks bisnis, dan konsistensi dengan fitur lain.

## **3.2. Weekend Stay Investigation**

In [17]:
# 1. Distribusi nilai di atas batas IQR (>5)
outlier_swn = df_raw[df_raw['stays_in_weekend_nights'] > 5].copy()
print(f"Jumlah baris dengan stays_in_weekend_nights > 5: {len(outlier_swn)}")
print(outlier_swn['stays_in_weekend_nights'].value_counts().sort_index())

# 2. Fokus ke nilai paling ekstrem (>10)
extreme_swn = df_raw[df_raw['stays_in_weekend_nights'] > 10].copy()
print(f"\nJumlah baris dengan stays_in_weekend_nights > 10: {len(extreme_swn)}")

# 3. Cross-check dengan market_segment & customer_type
print("\nDistribusi market_segment untuk outlier (>5):")
print(outlier_swn['market_segment'].value_counts(normalize=True) * 100)

print("\nDistribusi market_segment untuk seluruh dataset (pembanding):")
print(df_raw['market_segment'].value_counts(normalize=True) * 100)

print("\nDistribusi customer_type untuk outlier (>5):")
print(outlier_swn['customer_type'].value_counts(normalize=True) * 100)

print("\nDistribusi customer_type untuk seluruh dataset (pembanding):")
print(df_raw["customer_type"].value_counts(normalize=True) * 100)

# 4. Cross-check konsistensi dengan stays_in_week_nights
#    -> hitung rasio weekend:week, cek apakah proporsional/wajar
outlier_swn['ratio_weekend_to_week'] = (
    outlier_swn['stays_in_weekend_nights'] /
    outlier_swn['stays_in_week_nights'].replace(0, np.nan)
)
print("\nStatistik rasio weekend:week nights pada outlier:")
print(outlier_swn['ratio_weekend_to_week'].describe())

# 5. Curigai pola aneh: weekend_nights besar tapi week_nights = 0
#    -> tamu yang HANYA menginap saat weekend berkali-kali, tanpa weekday sama sekali
suspicious = df_raw[
    (df_raw['stays_in_weekend_nights'] > 6) &
    (df_raw['stays_in_week_nights'] == 0)
]
print(f"\nBaris mencurigakan (weekend_nights>6 tapi week_nights=0): {len(suspicious)}")
print(suspicious[['stays_in_weekend_nights', 'stays_in_week_nights',
                   'market_segment', 'customer_type', 'is_canceled']].head(20))

# 6. Lihat detail lengkap baris paling ekstrem (>=12 malam weekend) untuk inspeksi manual
top_extreme = df_raw[df_raw['stays_in_weekend_nights'] >= 12]
print(f"\nBaris dengan stays_in_weekend_nights >= 12: {len(top_extreme)}")
print(top_extreme[['hotel', 'market_segment', 'customer_type',
                    'stays_in_week_nights', 'stays_in_weekend_nights',
                    'adults', 'children', 'adr', 'is_canceled']])

# 7. Ringkasan
print("\n=== RINGKASAN ===")
print(f"Total baris > 5 (batas IQR): {len(outlier_swn)}")
print(f"Total baris > 10 (ekstrem): {len(extreme_swn)}")
print(f"Total baris mencurigakan (weekend besar, week=0): {len(suspicious)}")

Jumlah baris dengan stays_in_weekend_nights > 5: 265
stays_in_weekend_nights
6     153
7      19
8      60
9      11
10      7
12      5
13      3
14      2
16      3
18      1
19      1
Name: count, dtype: int64

Jumlah baris dengan stays_in_weekend_nights > 10: 15

Distribusi market_segment untuk outlier (>5):
market_segment
Online TA        42.264151
Offline TA/TO    21.886792
Groups           21.509434
Direct            9.433962
Corporate         4.528302
Complementary     0.377358
Name: proportion, dtype: float64

Distribusi market_segment untuk seluruh dataset (pembanding):
market_segment
Online TA        47.304632
Offline TA/TO    20.285619
Groups           16.593517
Direct           10.558673
Corporate         4.435045
Complementary     0.622330
Aviation          0.198509
Undefined         0.001675
Name: proportion, dtype: float64

Distribusi customer_type untuk outlier (>5):
customer_type
Transient          75.471698
Transient-Party    18.113208
Contract            6.415094
Na

### **Penjelasan**

Reservasi dengan lebih dari lima malam akhir pekan diperiksa untuk menentukan apakah durasi tinggi merupakan kesalahan pencatatan atau memang mencerminkan pola long stay yang valid.

### **Temuan**

- Sebanyak **265 reservasi** memiliki lebih dari lima malam akhir pekan. 
- Sebanyak **15 reservasi** di antaranya melebihi 12 malam akhir pekan dan termasuk kategori ekstrem.
- Tidak ditemukan reservasi dengan malam akhir pekan besar namun nol malam hari kerja, sehingga tidak ada indikasi kasus yang janggal secara struktural.
- Rasio weekend banding week nights pada outlier tergolong sempit, dengan rata rata 0,40 dan standar deviasi 0,049 pada rentang 0,32 hingga 0,60, yang menunjukkan pola durasi menginap yang cukup proporsional.
- Segmen **Groups** naik dari 16,59% (keseluruhan) menjadi 21,51% (outlier), kenaikan terbesar di antara semua segmen pasar. Sebaliknya, **Online TA** turun dari 47,30% menjadi 42,26%, menjadi under representasi paling besar. Sementara **Offline TA/TO** hanya berselisih tipis (20,29% menjadi 21,89%), sehingga belum cukup signifikan untuk disebut sebagai pola tersendiri.
- Customer type **Contract** naik dari 3,41% menjadi 6,42%, hampir dua kali lipat dan menjadi over representasi paling menonjol pada metrik ini. **Transient Party** turun dari 21,04% menjadi 18,11%, sementara **Transient** relatif stabil (75,06% menjadi 75,47%).

### **Insight**

Sebagian outlier durasi menginap kemungkinan merupakan long stay yang valid, didukung oleh rasio weekend banding week nights yang konsisten serta dua pola segmentasi yang cukup berbeda dari baseline, yaitu kenaikan proporsi **Groups** dan **Contract**. Kombinasi ini cukup masuk akal jika dikaitkan dengan reservasi grup atau kontrak jangka panjang, meskipun pola pada Offline TA/TO belum cukup kuat untuk dianggap sebagai indikasi tersendiri.

## **3.3. Weekday Stay Investigation**

In [18]:
# 1. Lihat sebaran nilai ekstrem (di atas batas IQR atas = 6)
outlier_swn = df_raw[df_raw['stays_in_week_nights'] > 6].copy()
print(f"Jumlah baris dengan stays_in_week_nights > 6: {len(outlier_swn)}")
print(outlier_swn['stays_in_week_nights'].value_counts().sort_index())

# 2. Fokus ke nilai yang benar-benar ekstrem (>30 malam, sesuai diskusi kita)
extreme_swn = df_raw[df_raw['stays_in_week_nights'] > 30].copy()
print(f"\nJumlah baris dengan stays_in_week_nights > 30: {len(extreme_swn)}")

# 3. Cross-check dengan market_segment & customer_type
#    -> untuk lihat apakah nilai besar terkonsentrasi di grup korporat/group booking
print("\nDistribusi market_segment untuk outlier (>6 malam):")
print(outlier_swn['market_segment'].value_counts(normalize=True) * 100)

print("\nDistribusi customer_type untuk outlier (>6 malam):")
print(outlier_swn['customer_type'].value_counts(normalize=True) * 100)

print("\nDistribusi market_segment untuk seluruh dataset (pembanding):")
print(df_raw['market_segment'].value_counts(normalize=True) * 100)

print("\nDistribusi customer_type untuk seluruh dataset (pembanding):")
print(df_raw["customer_type"].value_counts(normalize=True) * 100)

# 4. Cek konsistensi dengan stays_in_weekend_nights
#    -> curigai baris yang week_nights besar tapi weekend_nights = 0 (pola aneh)
suspicious = df_raw[
    (df_raw['stays_in_week_nights'] > 15) &
    (df_raw['stays_in_weekend_nights'] == 0)
]
print(f"\nBaris mencurigakan (week_nights>15 tapi weekend_nights=0): {len(suspicious)}")
print(suspicious[['stays_in_week_nights', 'stays_in_weekend_nights',
                   'market_segment', 'customer_type', 'is_canceled']].head(20))

# 5. Lihat detail lengkap baris paling ekstrem (>=40 malam) untuk inspeksi manual
top_extreme = df_raw[df_raw['stays_in_week_nights'] >= 40]
print(f"\nBaris dengan stays_in_week_nights >= 40: {len(top_extreme)}")
print(top_extreme[['hotel', 'market_segment', 'customer_type',
                    'stays_in_week_nights', 'stays_in_weekend_nights',
                    'adults', 'children', 'is_canceled']])

Jumlah baris dengan stays_in_week_nights > 6: 3354
stays_in_week_nights
7     1029
8      656
9      231
10    1036
11      56
12      42
13      27
14      35
15      85
16      16
17       4
18       6
19      44
20      41
21      15
22       7
24       3
25       6
26       1
30       5
32       1
33       1
34       1
35       1
40       2
41       1
42       1
50       1
Name: count, dtype: int64

Jumlah baris dengan stays_in_week_nights > 30: 9

Distribusi market_segment untuk outlier (>6 malam):
market_segment
Online TA        42.069171
Offline TA/TO    36.940966
Direct           11.538462
Groups            5.963029
Corporate         2.623733
Aviation          0.685748
Complementary     0.178891
Name: proportion, dtype: float64

Distribusi customer_type untuk outlier (>6 malam):
customer_type
Transient          71.496720
Contract           18.545021
Transient-Party     9.451401
Group               0.506857
Name: proportion, dtype: float64

Distribusi market_segment untuk seluru

### **Penjelasan**

Reservasi dengan lebih dari enam malam hari kerja diperiksa untuk menentukan apakah durasi menginap yang tinggi tersebut merupakan kesalahan pencatatan atau memang mencerminkan pola long stay yang valid.

### **Temuan**

- Sebanyak **3.354 reservasi** memiliki lebih dari enam malam hari kerja. 
- Sebanyak **9 reservasi** melebihi 30 malam, dan **5** di antaranya berada pada rentang 40 hingga 50 malam.
- Tidak ditemukan reservasi dengan lebih dari 15 malam hari kerja namun nol malam akhir pekan, sehingga tidak ada indikasi kasus "week only" yang janggal.
- Segmen **Offline TA/TO** naik dari 20,29% (keseluruhan) menjadi 36,94% (outlier), hampir dua kali lipat dan menjadi kenaikan paling signifikan. Sebaliknya, **Groups** turun dari 16,59% menjadi 5,96%, berlawanan arah dengan temuan pada outlier stays_in_weekend_nights sebelumnya, di mana Groups justru over represented. Segmen **Online TA** turun dari 47,30% menjadi 42,07%, konsisten under represented pada kedua analisis.
- Customer type **Contract** melonjak dari 3,41% menjadi 18,55%, sekitar 5,4 kali lipat, over representasi paling ekstrem dari seluruh metrik yang diperiksa sejauh ini. **Transient Party** turun cukup besar dari 21,04% menjadi 9,45%, sementara **Transient** relatif stabil (75,06% menjadi 71,50%).
- Dari lima baris paling ekstrem (≥40 malam), empat baris (index 9839, 14037, 14038, 33924) memiliki nilai adr wajar (25,50 hingga 110,50), berbeda dari beberapa outlier ekstrem pada stays_in_weekend_nights yang bernilai adr nol. Satu baris, index **101794**, memiliki adults sama dengan nol dan sebelumnya juga sudah ditandai sebagai anomali pada analisis stays_in_weekend_nights, sehingga kecurigaan terhadap baris ini semakin kuat karena muncul pada kedua daftar outlier.

### **Insight**

Pola pada outlier stays_in_week_nights berbeda secara struktural dari outlier stays_in_weekend_nights. Di sini, **Contract** dan **Offline TA/TO** yang mendominasi over representasi, sementara **Groups** justru under represented, kebalikan dari temuan pada malam akhir pekan. Pola ini cukup masuk akal jika dikaitkan dengan skenario bisnis nyata, misalnya kontrak korporat atau proyek jangka panjang yang biasanya dipesan melalui agen travel offline.

Nilai adr yang wajar pada mayoritas baris ekstrem membuat kelompok ini lebih meyakinkan sebagai long stay nyata dibandingkan beberapa kasus pada outlier stays_in_weekend_nights yang menunjukkan tanda data placeholder. Meski begitu, baris index 101794 tetap perlu ditandai sebagai kandidat anomali karena nilai adults yang tercatat nol.

Secara keseluruhan, mayoritas outlier pada stays_in_week_nights layak dipertahankan sebagai representasi long stay yang valid.

## **3.4. Adult Guest Investigation**

In [19]:
# 1. Distribusi nilai adults secara keseluruhan
print("Distribusi nilai adults:")
print(df_raw['adults'].value_counts().sort_index())

# 2. Kasus paling kritis: total tamu = 0 (adults=0 DAN children=0 DAN babies=0)
#    -> ini bukan soal outlier statistik, ini soal validitas booking
zero_guest = df_raw[
    (df_raw['adults'] == 0) &
    (df_raw['children'].fillna(0) == 0) &
    (df_raw['babies'] == 0)
]
print(f"\nJumlah baris dengan TOTAL TAMU = 0 (adults+children+babies): {len(zero_guest)}")
print(zero_guest[['hotel', 'market_segment', 'customer_type',
                   'stays_in_week_nights', 'stays_in_weekend_nights',
                   'reserved_room_type', 'adr', 'is_canceled']].head(20))

# 3. Kasus adults=0 TAPI ada children/babies (booking valid secara logis?)
#    -> booking anak/bayi tanpa pendamping dewasa itu janggal, perlu dicek
adults_zero_with_minor = df_raw[
    (df_raw['adults'] == 0) &
    ((df_raw['children'].fillna(0) > 0) | (df_raw['babies'] > 0))
]
print(f"\nJumlah baris adults=0 tapi ada children/babies: {len(adults_zero_with_minor)}")
print(adults_zero_with_minor[['adults', 'children', 'babies',
                               'market_segment', 'is_canceled']].head(20))

# 4. Nilai adults ekstrem (>10) -> kemungkinan data entry error
extreme_adults = df_raw[df_raw['adults'] > 10].copy()
print(f"\nJumlah baris dengan adults > 10: {len(extreme_adults)}")
print(extreme_adults[['hotel', 'adults', 'children', 'babies',
                       'market_segment', 'customer_type',
                       'reserved_room_type', 'assigned_room_type',
                       'stays_in_week_nights', 'is_canceled']])

# 5. Cross-check adults ekstrem dengan tipe kamar
#    -> kalau adults besar tapi room_type kecil (misal tipe A/B), itu makin mencurigakan
print("\nDistribusi reserved_room_type untuk adults > 10:")
print(extreme_adults['reserved_room_type'].value_counts())

print("\nDistribusi reserved_room_type untuk seluruh dataset (pembanding):")
print(df_raw['reserved_room_type'].value_counts(normalize=True) * 100)

# 6. Nilai adults yang "wajar tapi besar" (3-10) -> cek proporsinya
moderate_adults = df_raw[(df_raw['adults'] >= 3) & (df_raw['adults'] <= 10)]
print(f"\nJumlah baris adults 3-10 (kemungkinan keluarga besar/grup kecil, valid): {len(moderate_adults)}")
print(moderate_adults['adults'].value_counts().sort_index())

# 7. Ringkasan akhir: total baris yang perlu keputusan cleaning
print("\n=== RINGKASAN ===")
print(f"Total baris (zero guest, kandidat DROP): {len(zero_guest)}")
print(f"Total baris (adults=0 + ada anak/bayi, perlu cek manual): {len(adults_zero_with_minor)}")
print(f"Total baris (adults > 10, kandidat cap/drop): {len(extreme_adults)}")

Distribusi nilai adults:
adults
0       403
1     23027
2     89680
3      6202
4        62
5         2
6         1
10        1
20        2
26        5
27        2
40        1
50        1
55        1
Name: count, dtype: int64

Jumlah baris dengan TOTAL TAMU = 0 (adults+children+babies): 180
              hotel market_segment    customer_type  stays_in_week_nights  \
2224   Resort Hotel      Corporate  Transient-Party                     3   
2409   Resort Hotel      Corporate        Transient                     0   
3181   Resort Hotel         Groups  Transient-Party                     2   
3684   Resort Hotel         Groups  Transient-Party                     4   
3708   Resort Hotel         Groups  Transient-Party                     4   
4127   Resort Hotel  Offline TA/TO        Transient                     0   
9376   Resort Hotel  Offline TA/TO            Group                     0   
31765  Resort Hotel         Direct        Transient                     8   
32029  Resort H

### **Penjelasan**

Jumlah orang dewasa diperiksa bersama jumlah anak, bayi, tipe pelanggan, segmen pasar, dan durasi menginap untuk memisahkan group booking yang valid dari data yang secara logika tidak masuk akal.

### **Temuan**

- **180 reservasi** mempunyai total tamu nol.
- **223 reservasi** mencatat nol orang dewasa tetapi mempunyai anak atau bayi.
- **12 reservasi** mempunyai lebih dari 10 orang dewasa.
- Seluruh reservasi dengan lebih dari 10 orang dewasa bertipe `Group`.
- Seluruh 12 group booking besar tersebut dibatalkan.
- Sebanyak **6.268 reservasi** mempunyai 3–10 orang dewasa.

### **Insight**

Reservasi tanpa tamu tidak memiliki interpretasi operasional yang wajar sehingga layak dikeluarkan.

Sebaliknya, jumlah orang dewasa yang sangat besar memiliki pola group booking yang konsisten dan sebaiknya dipertahankan. Meskipun seluruh 12 booking tersebut dibatalkan, jumlah observasinya terlalu kecil untuk menyimpulkan bahwa group booking besar selalu berakhir dengan pembatalan.

## **3.5. Booking without Adult Guests**

In [20]:
adults_zero_minor = df_raw[
    (df_raw['adults'] == 0) &
    ((df_raw['children'].fillna(0) > 0) | (df_raw['babies'] > 0))
].copy()

# 1. Apakah children SELALU 2 di kasus ini?
print("Distribusi children pada kasus adults=0 + minor:")
print(adults_zero_minor['children'].value_counts())

# 2. Apakah terkonsentrasi di market_segment tertentu?
print("\nDistribusi market_segment pada kasus ini:")
print(adults_zero_minor['market_segment'].value_counts(normalize=True) * 100)

# 3. Cek adr -- apakah adr=0 juga (mirip pola zero_guest) atau normal?
print("\nStatistik adr pada kasus ini:")
print(adults_zero_minor['adr'].describe())

# 4. Cek apakah terkonsentrasi di rentang tanggal tertentu
#    -> kalau iya, kuat dugaan bug sistem yang muncul di periode tertentu
print("\nDistribusi arrival_date_year & month pada kasus ini:")
print(adults_zero_minor.groupby(['arrival_date_year', 'arrival_date_month']).size().sort_values(ascending=False).head(10))

# 5. Bandingkan dengan baris "normal" adults=2, children=0 dari segmen sama
#    -> untuk lihat apakah ini benar-benar swap adults<->children
normal_comparison = df_raw[
    (df_raw['adults'] == 2) &
    (df_raw['children'].fillna(0) == 0) &
    (df_raw['market_segment'] == 'Online TA')
]
print(f"\nJumlah baris 'normal' (adults=2, children=0, Online TA) sebagai pembanding: {len(normal_comparison)}")

Distribusi children pada kasus adults=0 + minor:
children
2.0    208
3.0     11
1.0      4
Name: count, dtype: int64

Distribusi market_segment pada kasus ini:
market_segment
Online TA        83.856502
Direct           13.004484
Complementary     1.793722
Offline TA/TO     1.345291
Name: proportion, dtype: float64

Statistik adr pada kasus ini:
count    223.000000
mean      81.168206
std       36.311836
min        0.000000
25%       73.740000
50%       88.770000
75%      104.150000
max      155.680000
Name: adr, dtype: float64

Distribusi arrival_date_year & month pada kasus ini:
arrival_date_year  arrival_date_month
2016               August                25
                   July                  18
2017               July                  15
2016               October               14
                   December              14
                   February              13
2017               April                 13
2016               September             11
                   Janu

### **Penjelasan**

Sebanyak 223 reservasi dengan `adults = 0` tetapi memiliki anak atau bayi diperiksa lebih lanjut untuk mengetahui apakah reservasi tersebut sepenuhnya tidak valid atau hanya mengalami kesalahan pencatatan.

### **Temuan**

- Mayoritas mencatat dua anak.
- Sebanyak **83,86%** berasal dari Online TA.
- Median ADR sebesar **88,77**.
- Kasus ditemukan pada beberapa bulan dan tahun yang berbeda.

### **Insight**

Tarif yang masuk akal, persebaran lintas periode, dan dominasi Online TA menunjukkan bahwa reservasi tersebut kemungkinan nyata, tetapi jumlah orang dewasa tidak tercatat dengan benar.

Mengubah `adults` menjadi satu merupakan koreksi minimal yang mempertahankan observasi. Namun, tindakan ini tetap harus dicatat sebagai asumsi karena jumlah orang dewasa sebenarnya tidak dapat dibuktikan dari dataset.

## **3.6. Children Investigation**

In [21]:
# 0. Cek missing value dulu -- ini terpisah dari isu outlier
print("Jumlah missing value di children:", df_raw['children'].isna().sum())
print(df_raw[df_raw['children'].isna()][['hotel', 'adults', 'babies', 'market_segment',
                                   'stays_in_week_nights', 'is_canceled']])

# 1. Distribusi nilai children (exclude NaN dulu biar rapi)
print("\nDistribusi nilai children:")
print(df_raw['children'].value_counts(dropna=False).sort_index())

# 2. Fokus ke nilai ekstrem: children = 10
extreme_children = df_raw[df_raw['children'] == 10].copy()
print(f"\nJumlah baris dengan children = 10: {len(extreme_children)}")
print(extreme_children[['hotel', 'adults', 'children', 'babies',
                         'market_segment', 'customer_type',
                         'reserved_room_type', 'assigned_room_type',
                         'stays_in_week_nights', 'adr', 'is_canceled']])

# 3. Nilai children = 3 -> masih masuk akal atau perlu dicurigai?
children_3 = df_raw[df_raw['children'] == 3]
print(f"\nJumlah baris dengan children = 3: {len(children_3)}")
print(children_3[['adults', 'children', 'babies', 'market_segment',
                   'reserved_room_type', 'adr']].head(15))

# 4. Cross-check children dengan adults -> rasio anak:dewasa yang wajar?
#    -> kalau children >> adults, itu janggal (siapa yang mengawasi?)
children_filled = df_raw['children'].fillna(0)
ratio_check = df_raw[children_filled > df_raw['adults']]
print(f"\nJumlah baris dengan children > adults: {len(ratio_check)}")
print(ratio_check[['adults', 'children', 'babies', 'market_segment', 'is_canceled']].head(15))

# 5. Cross-check dengan reserved_room_type
#    -> keluarga besar dengan banyak anak biasanya butuh room_type lebih besar
print("\nDistribusi reserved_room_type untuk children=10:")
print(extreme_children['reserved_room_type'].value_counts())

print("\nDistribusi reserved_room_type untuk seluruh dataset (pembanding):")
print(df_raw['reserved_room_type'].value_counts(normalize=True) * 100)

# 6. Cek apakah children=10 terkonsentrasi di segmen/waktu tertentu
print("\nDistribusi market_segment untuk children=10:")
print(extreme_children['market_segment'].value_counts())

print("\nDistribusi arrival_date_year & month untuk children=10:")
print(extreme_children.groupby(['arrival_date_year', 'arrival_date_month']).size())

# 7. Ringkasan
print("\n=== RINGKASAN ===")
print(f"Missing value: {df_raw['children'].isna().sum()}")
print(f"children=10 (kandidat cap/drop): {len(extreme_children)}")
print(f"children > adults (janggal, perlu cek): {len(ratio_check)}")

Jumlah missing value di children: 4
            hotel  adults  babies market_segment  stays_in_week_nights  \
40600  City Hotel       2       0      Undefined                     0   
40667  City Hotel       2       0         Direct                     2   
40679  City Hotel       3       0      Undefined                     2   
41160  City Hotel       2       0      Online TA                     5   

       is_canceled  
40600            1  
40667            1  
40679            1  
41160            1  

Distribusi nilai children:
children
0.0     110796
1.0       4861
2.0       3652
3.0         76
10.0         1
NaN          4
Name: count, dtype: int64

Jumlah baris dengan children = 10: 1
            hotel  adults  children  babies market_segment customer_type  \
328  Resort Hotel       2      10.0       0  Offline TA/TO      Contract   

    reserved_room_type assigned_room_type  stays_in_week_nights     adr  \
328                  D                  D                    10  133.

### **Penjelasan**

Distribusi jumlah anak dan konsistensinya dengan jumlah orang dewasa, tipe kamar, segmen pasar, serta karakteristik reservasi lainnya diperiksa sebelum menentukan tindakan cleaning.

### **Temuan**

- Sebagian besar reservasi tidak membawa anak.
- Sebanyak 4.861 reservasi membawa satu anak.
- Sebanyak 3.652 reservasi membawa dua anak.
- Sebanyak 76 reservasi membawa tiga anak.
- Hanya satu reservasi yang mencatat 10 anak.
- Terdapat 446 reservasi dengan jumlah anak lebih banyak daripada orang dewasa.

### **Insight**

Jumlah anak lebih banyak daripada jumlah orang dewasa tidak otomatis menunjukkan kesalahan; satu orang dewasa masih mungkin bepergian dengan dua anak.

Sebaliknya, kasus 10 anak hanya muncul satu kali dan tidak didukung pola group booking atau tipe kamar berkapasitas besar. Kasus tersebut layak diperlakukan sebagai anomali pencatatan.

## **3.7. Babies Investigation**

In [22]:
# 0. Cek missing value dulu -- ini terpisah dari isu outlier
print("Jumlah missing value di babies:", df_raw['babies'].isna().sum())
print(df_raw[df_raw['babies'].isna()][['hotel', 'adults', 'babies', 'market_segment',
                                   'stays_in_week_nights', 'is_canceled']])

# 1. Distribusi nilai babies (exclude NaN dulu biar rapi)
print("\nDistribusi nilai babies:")
print(df_raw['babies'].value_counts(dropna=False).sort_index())

# 2. Fokus ke nilai ekstrem: babies = 10
extreme_babies = df_raw[df_raw['babies'] == 10].copy()
print(f"\nJumlah baris dengan babies = 10: {len(extreme_babies)}")
print(extreme_babies[['hotel', 'adults', 'children', 'babies',
                         'market_segment', 'customer_type',
                         'reserved_room_type', 'assigned_room_type',
                         'stays_in_week_nights', 'adr', 'is_canceled']])

# 3. Nilai babies = 3 -> masih masuk akal atau perlu dicurigai?
babies_3 = df_raw[df_raw['babies'] == 3]
print(f"\nJumlah baris dengan babies = 3: {len(babies_3)}")
print(babies_3[['adults', 'children', 'babies', 'market_segment',
                   'reserved_room_type', 'adr']].head(15))

# 4. Cross-check babies dengan adults -> rasio anak:dewasa yang wajar?
#    -> kalau babies >> adults, itu janggal (siapa yang mengawasi?)
babies_filled = df_raw['babies'].fillna(0)
ratio_check = df_raw[babies_filled > df_raw['adults']]
print(f"\nJumlah baris dengan babies > adults: {len(ratio_check)}")
print(ratio_check[['adults', 'children', 'babies', 'market_segment', 'is_canceled']].head(15))

# 5. Cross-check dengan reserved_room_type
#    -> keluarga besar dengan banyak anak biasanya butuh room_type lebih besar
print("\nDistribusi reserved_room_type untuk babies=10:")
print(extreme_babies['reserved_room_type'].value_counts())

print("\nDistribusi reserved_room_type untuk seluruh dataset (pembanding):")
print(df_raw['reserved_room_type'].value_counts(normalize=True) * 100)

# 6. Cek apakah babies=10 terkonsentrasi di segmen/waktu tertentu
print("\nDistribusi market_segment untuk babies=10:")
print(extreme_babies['market_segment'].value_counts())

print("\nDistribusi arrival_date_year & month untuk babies=10:")
print(extreme_babies.groupby(['arrival_date_year', 'arrival_date_month']).size())

# 7. Ringkasan
print("\n=== RINGKASAN ===")
print(f"Missing value: {df_raw['babies'].isna().sum()}")
print(f"babies=10 (kandidat cap/drop): {len(extreme_babies)}")
print(f"babies > adults (janggal, perlu cek): {len(ratio_check)}")

Jumlah missing value di babies: 0
Empty DataFrame
Columns: [hotel, adults, babies, market_segment, stays_in_week_nights, is_canceled]
Index: []

Distribusi nilai babies:
babies
0     118473
1        900
2         15
9          1
10         1
Name: count, dtype: int64

Jumlah baris dengan babies = 10: 1
            hotel  adults  children  babies market_segment customer_type  \
46619  City Hotel       2       0.0      10      Online TA     Transient   

      reserved_room_type assigned_room_type  stays_in_week_nights    adr  \
46619                  D                  D                     2  84.45   

       is_canceled  
46619            0  

Jumlah baris dengan babies = 3: 0
Empty DataFrame
Columns: [adults, children, babies, market_segment, reserved_room_type, adr]
Index: []

Jumlah baris dengan babies > adults: 5
       adults  children  babies market_segment  is_canceled
46150       0       2.0       1      Online TA            0
46619       2       0.0      10      Online TA    

### **Penjelasan**

Nilai `babies` diperiksa untuk membedakan reservasi keluarga yang wajar dari jumlah bayi yang sangat tidak masuk akal.

### **Temuan**

- Sebanyak 900 reservasi mencatat satu bayi.
- Sebanyak 15 reservasi mencatat dua bayi.
- Kasus 9 dan 10 bayi masing-masing hanya muncul satu kali.

### **Insight**

Satu atau dua bayi masih merupakan komposisi tamu yang masuk akal. Sebaliknya, nilai 9 dan 10 bayi sangat terisolasi dan tidak menunjukkan pola group booking yang dapat menjelaskannya. Kedua baris tersebut layak dikeluarkan sebagai kemungkinan kesalahan input.

## **3.8. Waiting List Investigation**

In [23]:
# 0. Cek missing value
print("Jumlah missing value:", df_raw['days_in_waiting_list'].isna().sum())

# 1. Distribusi umum -> berapa besar proporsi yang benar-benar 0 vs >0
zero_wait = (df_raw['days_in_waiting_list'] == 0).sum()
nonzero_wait = (df_raw['days_in_waiting_list'] > 0).sum()
print(f"\nJumlah days_in_waiting_list = 0: {zero_wait} ({zero_wait/len(df_raw)*100:.1f}%)")
print(f"Jumlah days_in_waiting_list > 0: {nonzero_wait} ({nonzero_wait/len(df_raw)*100:.1f}%)")

# 2. Statistik deskriptif untuk subset yang > 0 saja (lebih informatif dari full data)
nonzero_df_raw = df_raw[df_raw['days_in_waiting_list'] > 0]
print("\nStatistik days_in_waiting_list (hanya yang > 0):")
print(nonzero_df_raw['days_in_waiting_list'].describe())

# 3. Cross-check dengan market_segment -> uji hipotesis "ini pola Groups"
print("\nDistribusi market_segment untuk waiting_list > 0:")
print(nonzero_df_raw['market_segment'].value_counts(normalize=True) * 100)

print("\nDistribusi market_segment untuk seluruh dataset (pembanding):")
print(df_raw['market_segment'].value_counts(normalize=True) * 100)

# 4. Cross-check dengan customer_type
print("\nDistribusi customer_type untuk waiting_list > 0:")
print(nonzero_df_raw['customer_type'].value_counts(normalize=True) * 100)

# 5. Cek hubungan dengan is_canceled -> apakah waiting list lama = kecenderungan cancel?
print("\nRata-rata is_canceled untuk waiting_list=0 vs >0:")
print(df_raw.groupby(df_raw['days_in_waiting_list'] > 0)['is_canceled'].mean())

# 6. Fokus ke nilai paling ekstrem (>200 hari)
extreme_wait = df_raw[df_raw['days_in_waiting_list'] > 200].copy()
print(f"\nJumlah baris dengan days_in_waiting_list > 200: {len(extreme_wait)}")
print(extreme_wait[['hotel', 'market_segment', 'customer_type',
                     'days_in_waiting_list', 'lead_time',
                     'is_canceled', 'adr']].sort_values('days_in_waiting_list', ascending=False))

# 7. Cek korelasi kasar dengan lead_time -> wajar kalau waiting list lama
#    terjadi pada booking dengan lead_time panjang juga
print("\nKorelasi days_in_waiting_list vs lead_time (subset >0):")
print(nonzero_df_raw[['days_in_waiting_list', 'lead_time']].corr())

# 8. Cek days_in_waiting_list > lead_time
print(f"\nKondisi saat days_in_waiting_list lebih lama dibandingkan lead_time: {(df_raw["days_in_waiting_list"] > df_raw["lead_time"]).sum()}")

# 9. Ringkasan
print("\n=== RINGKASAN ===")
print(f"Total baris waiting_list > 0: {nonzero_wait}")
print(f"Total baris waiting_list > 200 (ekstrem): {len(extreme_wait)}")

Jumlah missing value: 0

Jumlah days_in_waiting_list = 0: 115692 (96.9%)
Jumlah days_in_waiting_list > 0: 3698 (3.1%)

Statistik days_in_waiting_list (hanya yang > 0):
count    3698.000000
mean       74.938345
std        67.482918
min         1.000000
25%        38.000000
50%        56.000000
75%        91.000000
max       391.000000
Name: days_in_waiting_list, dtype: float64

Distribusi market_segment untuk waiting_list > 0:
market_segment
Offline TA/TO    56.165495
Groups           42.130882
Corporate         1.189832
Direct            0.378583
Online TA         0.081125
Complementary     0.054083
Name: proportion, dtype: float64

Distribusi market_segment untuk seluruh dataset (pembanding):
market_segment
Online TA        47.304632
Offline TA/TO    20.285619
Groups           16.593517
Direct           10.558673
Corporate         4.435045
Complementary     0.622330
Aviation          0.198509
Undefined         0.001675
Name: proportion, dtype: float64

Distribusi customer_type untuk w

### **Penjelasan**

Fitur `days_in_waiting_list` diperiksa untuk memahami distribusinya dan melihat hubungan awal dengan pembatalan reservasi.

### **Temuan**

- Sebanyak **96,9%** reservasi tidak masuk waiting list.
- Hanya **3,1% atau 3.698 reservasi** mempunyai waktu tunggu.
- Median waktu tunggu pada kelompok tersebut adalah **56 hari**.
- Offline TA/TO mencakup **56,17%** kelompok waiting list.
- Groups mencakup **42,13%** kelompok waiting list.
- Cancellation rate tanpa waiting list: **36,19%**.
- Cancellation rate dengan waiting list: **63,79%**.
- Korelasi waiting time dengan lead time sebesar **0,589**.

### **Insight**

Reservasi yang masuk waiting list mempunyai cancellation rate sekitar **27,6 poin persentase lebih tinggi** daripada reservasi tanpa waiting list. Fitur ini berpotensi menjadi indikator risiko yang kuat.

Namun, hasil tersebut masih berupa hubungan deskriptif. Analisis ini belum membuktikan bahwa waiting list menyebabkan pembatalan karena kelompok tersebut juga memiliki karakteristik lain, seperti lead time, market segment, dan customer type yang berbeda.

## **3.9. Average Daily Rate Investigation**

In [24]:
# 0. Cek missing value & statistik dasar
print("Jumlah missing value:", df_raw['adr'].isna().sum())
print("\nStatistik deskriptif adr:")
print(df_raw['adr'].describe())

# 1. KASUS 1: nilai negatif -> secara logis tidak valid
negative_adr = df_raw[df_raw['adr'] < 0].copy()
print(f"\nJumlah baris dengan adr < 0: {len(negative_adr)}")
print(negative_adr[['hotel', 'market_segment', 'customer_type',
                     'adults', 'children', 'babies',
                     'reserved_room_type', 'adr', 'is_canceled']])

# 2. KASUS 2: adr = 0 -> perlu dicek, apakah ini overlap dengan zero-guest
#    yang sudah kita drop, atau ada kasus lain (misal complementary stay)
zero_adr = df_raw[df_raw['adr'] == 0]
print(f"\nJumlah baris dengan adr = 0: {len(zero_adr)}")
print("Distribusi market_segment untuk adr=0:")
print(zero_adr['market_segment'].value_counts())
print("\nDistribusi customer_type untuk adr=0:")
print(zero_adr['customer_type'].value_counts())
# cek overlap dengan zero-guest yang sudah didrop sebelumnya
zero_adr_zero_guest_overlap = zero_adr[
    (zero_adr['adults'] == 0) & (zero_adr['children'].fillna(0) == 0) & (zero_adr['babies'] == 0)
]
print(f"Overlap dengan zero-guest (sudah ter-drop di tahap adults): {len(zero_adr_zero_guest_overlap)}")
print(f"Sisa adr=0 DI LUAR zero-guest (perlu investigasi terpisah): {len(zero_adr) - len(zero_adr_zero_guest_overlap)}")

# 3. KASUS 3: nilai maximum ekstrem (5400) -> investigasi baris spesifik
max_adr = df_raw[df_raw['adr'] == df_raw['adr'].max()]
print(f"\nBaris dengan adr maksimum ({df_raw['adr'].max()}):")
print(max_adr[['hotel', 'market_segment', 'customer_type', 'adults', 'children',
                'reserved_room_type', 'assigned_room_type', 'stays_in_week_nights',
                'stays_in_weekend_nights', 'lead_time', 'is_canceled']])

# 4. Lihat top 20 adr tertinggi -> apakah 5400 itu benar-benar terisolasi
#    atau ada beberapa baris tinggi lain yang mirip (pola vs anomali tunggal)
top20_adr = df_raw.nlargest(20, 'adr')
print("\nTop 20 nilai adr tertinggi:")
print(top20_adr[['hotel', 'market_segment', 'reserved_room_type', 'adr', 'is_canceled']])

# 5. Cross-check adr tinggi (di luas batas IQR atas ~211) dengan reserved_room_type
#    -> apakah adr tinggi konsisten dengan tipe kamar premium?
high_adr = df_raw[df_raw['adr'] > 211].copy()
print(f"\nJumlah baris dengan adr > 211 (batas IQR atas): {len(high_adr)}")
print("\nDistribusi reserved_room_type untuk adr > 211:")
print(high_adr['reserved_room_type'].value_counts(normalize=True) * 100)
print("\nDistribusi reserved_room_type untuk seluruh dataset (pembanding):")
print(df_raw['reserved_room_type'].value_counts(normalize=True) * 100)

# 6. Rata-rata adr per reserved_room_type -> baseline wajar per tipe kamar
print("\nRata-rata adr per reserved_room_type (seluruh data):")
print(df_raw.groupby('reserved_room_type')['adr'].agg(['mean', 'median', 'std', 'max']).sort_values('mean'))

# 7. Ringkasan
print("\n=== RINGKASAN ===")
print(f"adr < 0 (kandidat DROP): {len(negative_adr)}")
print(f"adr = 0 di luar zero-guest (perlu cek manual): {len(zero_adr) - len(zero_adr_zero_guest_overlap)}")
print(f"adr = max (5400) (kandidat investigasi/drop): {len(max_adr)}")
print(f"adr > 211 total (JANGAN otomatis dianggap error): {len(high_adr)}")

Jumlah missing value: 0

Statistik deskriptif adr:
count    119390.000000
mean        101.831122
std          50.535790
min          -6.380000
25%          69.290000
50%          94.575000
75%         126.000000
max        5400.000000
Name: adr, dtype: float64

Jumlah baris dengan adr < 0: 1
              hotel market_segment    customer_type  adults  children  babies  \
14969  Resort Hotel         Groups  Transient-Party       2       0.0       0   

      reserved_room_type   adr  is_canceled  
14969                  A -6.38            0  

Jumlah baris dengan adr = 0: 1959
Distribusi market_segment untuk adr=0:
market_segment
Complementary    680
Online TA        367
Offline TA/TO    332
Groups           252
Direct           240
Corporate         82
Aviation           6
Name: count, dtype: int64

Distribusi customer_type untuk adr=0:
customer_type
Transient          1450
Transient-Party     452
Group                33
Contract             24
Name: count, dtype: int64
Overlap dengan 

### **Penjelasan**

ADR diperiksa untuk membedakan tarif yang valid, tarif gratis, dan nilai yang kuat diduga sebagai kesalahan pencatatan.

### **Temuan**

- Ditemukan satu ADR negatif sebesar **−6,38**.
- Sebanyak **1.959 reservasi** memiliki ADR nol.
- Sebanyak 680 ADR nol berasal dari segmen Complementary.
- ADR maksimum mencapai **5.400**.
- Nilai tertinggi kedua hanya **510**.
- Sebanyak **3.794 reservasi** memiliki ADR di atas batas IQR sebesar 211.
- ADR tinggi banyak ditemukan pada room type premium seperti G, F, E, dan H.

### **Insight**

ADR tinggi tidak otomatis merupakan error karena dapat dijelaskan oleh tipe kamar premium. Oleh karena itu, sebagian besar nilai di atas batas IQR tetap dipertahankan.

ADR negatif tidak memiliki interpretasi tarif normal, sedangkan ADR 5.400 lebih dari sepuluh kali nilai tertinggi berikutnya dan sangat terisolasi. Kedua nilai tersebut mempunyai indikasi kuat sebagai kesalahan data.

ADR nol juga tidak otomatis salah karena sebagian kasus merupakan complimentary stay. Namun, penyebab ADR nol di luar segmen Complementary tetap menjadi keterbatasan yang perlu didokumentasikan.

## **3.10. Outlier Treatment Decision**

##### Keputusan Penanganan Outlier:

- Fitur `lead_time` memiliki nilai tidak wajar yaitu sebesar 737 hari. Namun nilai tersebut tetap masuk akal karena sebagian kecil orang memang melakukan booking pada jauh-jauh hari, sehingga nilai tersebut dapat dipertahankan.

- Fitur `arrival_date_week_number` dan `arrival_date_day_of_month` tidak memiliki anomali sama sekali, jadi tidak memerlukan penanganan apapun.

- Fitur `stays_in_weekend_nights` memiliki nilai yang dianggap tidak wajar yaitu > 5 minggu, sesuai dengan batas atas dari fitur tersebut.Perlu investigasi untuk memeriksa apakah nilai tidak wajar tersebut dipengaruhi karena pemesanan oleh corporate dan lain sebagainya. Berdasarkan hasil investigasi, diambil keputusan bahwa nilai tidak wajar tersebut layak untuk dipertahankan.

- Fitur `stays_in_week_nights` memerlukan investigasi lebih lanjut dikarenakan banyak nilai tidak wajar, seperti menginap lebih dari 30 hari. Perlu investigasi untuk memeriksa apakah nilai tidak wajar tersebut dipengaruhi karena pemesanan oleh corporate dan lain sebagainya. Berdasarkan hasil investigasi, diambil keputusan bahwa nilai tidak wajar tersebut layak untuk dipertahankan.

- Fitur `adults` memiliki nilai yang dianggap tidak wajar, yaitu 0 karena artinya tidak ada tamu sama sekali. Namun perlu dilakukan evaluasi lanjutan untuk memeriksa apakah booking tanpa orang dewasa tersebut dikarenakan adanya booking dengan children. Sebagai keputusan, booking yang tidak terdapat orang dewasa, anak-anak, dan bayi akan dihapus, untuk booking yang tidak terdapat orang dewasa namun terdapat anak-anak dan bayi akan mengubah nilai orang dewasa menjadi 1 (`adults`=1), dan orang dewasa > 10 orang akan dipertahankan.

- Fitur `children` memiliki nilai yang dianggap tidak wajar yaitu 10 karena nilai tersebut tidak realistis kecuali ada konteks khusus, sehingga perlu dilakukan investigasi lebih lanjut. Berdasarkan hasil investigasi, hanya ditemukan sebuah booking dengan 10 anak-anak, sehingga booking tersebut dapat di drop.

- Fitur `babies` memiliki nilai yang dianggap tidak wajar yaitu 9 dan 10 karena nilai tersebut tidak realistis kecuali ada konteks khusus, sehingga perlu dilakukan investigasi lebih lanjut. Berdasarkan hasil investigasi, hanya ditemukan masing-masing sebuah booking dengan 9 dan 10 anak-anak, sehingga booking tersebut dapat di drop.

- Semua nilai pada fitur `previous_cancellations` dapat dipertahankan karena tidak ada nilai yang dianggap sebagai nilai yang tidak wajar.

- Semua nilai pada fitur `previous_bookings_not_canceled` dapat dipertahankan karena tidak ada nilai yang dianggap sebagai nilai yang tidak wajar.

- Semua nilai pada fitur `booking_changes` dapat dipertahankan karena tidak ada nilai yang dianggap sebagai nilai yang tidak wajar.

- Semua nilai pada fitur `days_in_waiting_list` dapat dipertahankan karena tidak ada nilai yang dianggap sebagai nilai yang tidak wajar. Tidak ada pelanggaran `days_in_waiting list > lead_time`.

- Fitur `adr` memilliki nilai tidak wajar, yaitu 0 dan 5400 sehingga nilai tersebut dapat didrop.

- Semua nilai pada fitur `required_car_parking_spaces` dapat dipertahankan karena tidak ada nilai yang dianggap sebagai nilai yang tidak wajar.

- Semua nilai pada fitur `total_of_special_requests` dapat dipertahankan karena tidak ada nilai yang dianggap sebagai nilai yang tidak wajar.

# **Section 4: Data Cleaning**

## **4.1. Copy Datasets**
Membuat salinan dataset untuk hasil cleaning.

In [25]:
df = df_raw.copy()

## **4.2. Remove Leakage Features**

In [26]:
confirmed_leakage_features = [
    "reservation_status",
    "reservation_status_date"
]
df.drop(
    columns=confirmed_leakage_features,
    inplace=True
)

print("Removed leakage features:")
print(confirmed_leakage_features)

print("\nRemaining dataset shape:")
print(df.shape)

Removed leakage features:
['reservation_status', 'reservation_status_date']

Remaining dataset shape:
(119390, 30)


### **Penjelasan**

`reservation_status` dan `reservation_status_date` dihapus karena keduanya merepresentasikan informasi yang terbentuk setelah hasil akhir reservasi diketahui.

### **Insight**

Menggunakan kedua fitur tersebut dapat membuat model tampak sangat akurat karena model secara tidak langsung telah mengetahui apakah reservasi berakhir sebagai canceled, no-show, atau check-out.

Penghapusan leakage membuat evaluasi lebih realistis dan memastikan model hanya belajar dari informasi yang tersedia sebelum hasil akhir diketahui.

## **4.3. Standarize Undefined Categories**

### 4.3.1 Meal Feature With Undefined Value Will Be Replaced With Unknown
| Value       | Meaning                                                          |
| ----------- | ---------------------------------------------------------------- |
| `BB`        | **Bed & Breakfast** — breakfast included                         |
| `HB`        | **Half Board** — breakfast + one additional meal, usually dinner |
| `FB`        | **Full Board** — breakfast + lunch + dinner                      |
| `SC`        | **Self Catering** — no meals included                            |
| `Undefined` | Will Be Replaced With Unknown                                    |


In [27]:
df["meal"] = df["meal"].replace(
    "Undefined", "Unknown"
)

### 4.3.2 Distribution Channel Undefined Value Will Be Replaced With Unknown
| Value       | Meaning                                                              |
| ----------- | -------------------------------------------------------------------- |
| `Direct`    | Guest booked directly with the hotel                                 |
| `Corporate` | Booking from a corporate/company account                             |
| `TA/TO`     | **Travel Agent / Tour Operator**                                     |
| `GDS`       | **Global Distribution System**, e.g. systems used by travel agencies |
| `Undefined` | Will Be Replaced With Unknown                                        |


In [28]:
df["distribution_channel"] = df["distribution_channel"].replace(
    "Undefined", "Unknown"
)

### 4.3.3 Market Segment Undefined Value Will Be Replaced With Unknown

In [29]:
df["market_segment"] = df["market_segment"].replace(
    "Undefined", "Unknown"
)

### **Penjelasan**

Kategori `Undefined` pada `meal`, `market_segment`, dan `distribution_channel` diganti menjadi `Unknown`. Standardisasi ini dilakukan agar nilai yang tidak diketahui mempunyai label yang konsisten.

### **Insight**

Perubahan ini tidak mengisi informasi yang hilang dengan kategori bisnis tertentu. Tujuannya adalah mempertahankan ketidakpastian sebagai informasi eksplisit dan mencegah model menganggap `Undefined` sebagai kategori yang berbeda dari `Unknown`.

## **4.4 Children NaN Values Will Be Fill With 0 & Standardize As Int64**


In [30]:
df["children"] = df["children"].fillna(0.0)
df["children"] = df["children"].astype("int64")

### **Penjelasan**

Empat missing value pada `children` diisi dengan nol dan tipe datanya diubah menjadi integer.

### **Insight**

Dampak kuantitatifnya sangat kecil karena hanya memengaruhi sekitar 0,003% data. Namun, pengisian nol tetap menggunakan asumsi bahwa missing berarti tamu tidak membawa anak. Asumsi ini perlu dicatat sebagai bagian dari dokumentasi preprocessing.

## **4.5 Agent Will Be Having Prefix Of "agent_" And Replace 0 Value As No Agent**


In [31]:
df["agent"] = df["agent"].fillna(0).astype("int64")

df["agent"] = df["agent"].astype(str)

df["agent"] = "agent_" + df["agent"]

df["agent"] = df["agent"].replace("agent_0", "no agent")

### **Penjelasan**

Missing value pada `agent` diisi dengan nol, kemudian kode agent diubah menjadi teks dengan prefix `agent_`. Nilai `agent_0` selanjutnya diubah menjadi `no agent`.

### **Insight**

Transformasi ini mencegah model memperlakukan kode agent sebagai angka berurutan. Kategori `no agent` juga mempertahankan informasi bahwa booking dilakukan tanpa agen, yang dapat memiliki pola pembatalan berbeda dari booking melalui perantara.

## **4.6 Company Will Be Having Prefix Of "company_" And Replace 0 Value As No Company**


In [32]:
df["company"] = df["company"].fillna(0).astype("int64")

df["company"] = df["company"].astype(str)

df["company"] = "company_" + df["company"]

df["company"] = df["company"].replace("company_0", "no company")

### **Penjelasan**

Missing value pada `company` diperlakukan sebagai tidak adanya perusahaan, kemudian ID diubah menjadi kategori dengan prefix `company_`.

### **Insight**

Meskipun missing value mencapai 94,31%, fitur ini tidak otomatis harus dihapus. Tingginya missing dapat mencerminkan bahwa sebagian besar reservasi berasal dari pelanggan nonkorporat. Fitur `has_company` yang dibuat kemudian dapat merangkum informasi tersebut secara lebih sederhana.> `company` aslinya ID numerik (bukan besaran kuantitatif), jadi mengubahnya jadi string kategorikal itu benar secara konsep. NaN diasumsikan berarti tidak pakai `company`.

## **4.7 Country NaN Values Will Be Filled With Unknown**

In [33]:
print('Distribusi distribution_channel untuk baris country NaN:')
print(df[df["country"].isna()]["distribution_channel"].value_counts(normalize=True) * 100)
print()
print('Distribusi distribution_channel untuk seluruh dataset:')
print(df["distribution_channel"].value_counts(normalize=True) * 100)

Distribusi distribution_channel untuk baris country NaN:
distribution_channel
Corporate    38.114754
Direct       33.196721
TA/TO        28.688525
Name: proportion, dtype: float64

Distribusi distribution_channel untuk seluruh dataset:
distribution_channel
TA/TO        81.975040
Direct       12.266521
Corporate     5.592596
GDS           0.161655
Unknown       0.004188
Name: proportion, dtype: float64


In [34]:
df["country"] = df["country"].fillna("Unknown")

### **Insight:**
 `country` hanya memiliki missing value sekitar 0.41% dari data, tapi ternyata **tidak acak**: baris dengan `country` NaN terkonsentrasi di `distribution_channel = Corporate` (38% vs 5.6% di populasi keseluruhan) dan `Direct` (33% vs 12.3% di populasi), sementara `TA/TO` sangat kurang terwakili (29% vs 82% di populasi). Polanya masuk akal secara bisnis: booking lewat OTA/travel agent (`TA/TO`) hampir selalu mencatat negara asal tamu, sedangkan booking langsung/corporate lebih longgar soal kelengkapan data ini.

Karena pola missing-nya sistematis (bukan MCAR) tapi volumenya kecil (0.41%), diisi dengan kategori eksplisit `"Unknown"` karena modus akan salah merepresentasikan baris yang kemungkinan besar berasal dari channel non-OTA tadi (bukan benar-benar dari Portugal), sementara `"Unknown"` menjaga sinyal "data tidak tercatat" tetap terlihat oleh model, alih-alih disamarkan jadi kategori lain.

## **4.8 Clean Invalid Adult Record**

In [35]:
# 1. DROP: zero-guest total (adults=0, children=0, babies=0)
#    -> record tidak valid: 0 tamu + adr=0, bukan stay nyata
zero_guest_mask = (
    (df['adults'] == 0) &
    (df['children'].fillna(0) == 0) &
    (df['babies'] == 0)
)
print(f"Drop zero-guest: {zero_guest_mask.sum()} baris")
df = df[~zero_guest_mask]

# 2. IMPUTE: adults=0 tapi ada children/babies -> set adults=1 (minimal plausible)
#    -> TIDAK melakukan swap, hanya koreksi nilai implausible ke nilai minimal
invalid_adult_mask = (
    (df['adults'] == 0) &
    ((df['children'].fillna(0) > 0) | (df['babies'] > 0))
)
print(f"Impute adults=1: {invalid_adult_mask.sum()} baris")
df.loc[invalid_adult_mask, 'adults'] = 1

# 3. KEEP: adults > 10 (terverifikasi pola group booking konsisten: 
#    customer_type=Group, is_canceled=1, market_segment Direct/Offline TA/TO)
#    -> tidak ada modifikasi

Drop zero-guest: 180 baris
Impute adults=1: 223 baris


## **4.9 Clean Invalid Children Record**

In [36]:
# KEEP: children 0, 1, 2, 3 -> semua terverifikasi wajar (lihat room_type check untuk children=3)
# tidak perlu modifikasi

# DROP: children = 10 -> kejadian tunggal, room_type tidak konsisten dengan
# kapasitas fisik (D bukan large room), tidak ada pola pendukung (n=1)
invalid_children_mask = df['children'] == 10
print(f"Baris dengan children=10 (akan di-drop): {invalid_children_mask.sum()}")
df = df[~invalid_children_mask]

# Verifikasi hasil akhir
print(f"\nDistribusi children setelah cleaning:")
print(df['children'].value_counts().sort_index())
print(f"\nTotal baris setelah cleaning children: {len(df)}")

Baris dengan children=10 (akan di-drop): 1

Distribusi children setelah cleaning:
children
0    110620
1      4861
2      3652
3        76
Name: count, dtype: int64

Total baris setelah cleaning children: 119209


### **Penjelasan**

Satu reservasi dengan 10 anak dihapus karena merupakan kasus tunggal dan tidak konsisten dengan kapasitas kamar maupun pola group booking.

### **Insight**

Cleaning hanya diterapkan pada kasus dengan bukti anomali yang kuat. Reservasi dengan satu sampai tiga anak tetap dipertahankan karena masih mempunyai interpretasi bisnis yang wajar.

## **4.10 Clean Invalid Babies Record**

In [37]:
# DROP: babies = 9 atau 10 -> kejadian tunggal (n=1 masing-masing), tidak ada
# pola pendukung, implausible secara fisik (rasio bayi:dewasa mustahil diawasi),
# tidak seperti kasus adults>10 yang punya 12 baris berpola konsisten
invalid_babies_mask = df['babies'].isin([9, 10])
print(f"Baris dengan babies=9 atau 10 (akan di-drop): {invalid_babies_mask.sum()}")
df = df[~invalid_babies_mask]

# Catatan: baris adults=0 & babies=1 (3 baris) SUDAH tertangani otomatis
# lewat aturan impute adults=1 yang sudah diterapkan sebelumnya di tahap
# cleaning fitur adults -- tidak perlu aturan tambahan di sini.
# (Verifikasi cepat: cek apakah masih ada baris begini di df)
check_remaining = df[(df['adults'] == 0) & (df['babies'] > 0)]
print(f"Sisa baris adults=0 & babies>0 setelah cleaning adults: {len(check_remaining)}")

# Verifikasi hasil akhir
print(f"\nDistribusi babies setelah cleaning:")
print(df['babies'].value_counts().sort_index())
print(f"\nTotal baris setelah cleaning babies: {len(df)}")

Baris dengan babies=9 atau 10 (akan di-drop): 2
Sisa baris adults=0 & babies>0 setelah cleaning adults: 0

Distribusi babies setelah cleaning:
babies
0    118292
1       900
2        15
Name: count, dtype: int64

Total baris setelah cleaning babies: 119207


### **Penjelasan**

Dua reservasi yang mencatat 9 atau 10 bayi dihapus. Reservasi dengan satu atau dua bayi tetap dipertahankan.

### **Insight**

Pendekatan ini menghindari penghapusan massal berdasarkan IQR dan hanya mengeluarkan nilai yang sangat terisolasi serta sulit dijelaskan secara operasional.

## **4.11 Clean Invalid ADR Record**

In [38]:
# 1. DROP: adr < 0 -> tidak ada interpretasi bisnis valid untuk tarif negatif
negative_adr_mask = df['adr'] < 0
print(f"Drop adr negatif: {negative_adr_mask.sum()} baris")
df = df[~negative_adr_mask]

# 2. DROP: adr = 5400 -> terisolasi ekstrem (gap 10.6x dari nilai tertinggi
#    kedua yaitu 510), 129 std dev di atas mean room type A, pola data entry error
extreme_adr_mask = df['adr'] == 5400
print(f"Drop adr=5400: {extreme_adr_mask.sum()} baris")
df = df[~extreme_adr_mask]

# 3. KEEP: adr = 0
#    - 91.5% dari kasus Complementary market_segment -> terjelaskan penuh (stay gratis)
#    - sisanya (1130 baris) tidak ada bukti kuat untuk drop, KEEP dengan catatan
#      limitation eksplisit di dokumentasi (penyebab pasti tidak diketahui)
#    -> tidak ada modifikasi kode, cukup dicatat di notebook

# 4. KEEP: adr > 211 (di luar 2 kasus di atas)
#    -> terkonfirmasi driven by room type premium (G/F/D/E), bukan anomali
#    -> tidak ada modifikasi kode

# Verifikasi hasil akhir
print(f"\nStatistik adr setelah cleaning:")
print(df['adr'].describe())
print(f"\nTotal baris setelah cleaning adr: {len(df)}")

Drop adr negatif: 1 baris
Drop adr=5400: 1 baris

Statistik adr setelah cleaning:
count    119205.000000
mean        101.925500
std          48.042823
min           0.000000
25%          69.500000
50%          94.950000
75%         126.000000
max         510.000000
Name: adr, dtype: float64

Total baris setelah cleaning adr: 119205


### **Penjelasan**

Satu ADR negatif dan satu ADR sebesar 5.400 dihapus. ADR nol dan ADR tinggi lainnya tetap dipertahankan karena masih mempunyai kemungkinan penjelasan bisnis.

### **Temuan Setelah Cleaning**

- Jumlah data akhir tahap ini: **119.205 baris**.
- ADR minimum menjadi **0**.
- ADR maksimum menjadi **510**.
- Median ADR menjadi **94,95**.
- Rata-rata ADR menjadi **101,93**.

### **Insight**

Penghapusan hanya dua baris memperbaiki rentang ADR tanpa menghilangkan variasi tarif kamar yang valid. Nilai rata-rata dan median juga tetap relatif stabil, menunjukkan

# **Section 5: Feature Engineering**

Fitur baru dibuat dengan menggabungkan fitur yang ada atau mengubah informasi yang ada agar lebih baik mewakili perilaku pelanggan dan karakteristik pemesanan. 
| Fitur | Deskripsi | 
|---|---| 
| `total_stay_nights` | Total jumlah malam dalam pemesanan, dihitung dari malam akhir pekan dan malam hari kerja. |
| `total_guests` | Total jumlah tamu, termasuk dewasa, anak-anak, dan bayi. | 
| `has_children` | Menunjukkan apakah pemesanan termasuk satu atau lebih anak-anak. | 
| `has_babies` | Menunjukkan apakah pemesanan termasuk satu atau lebih bayi. | 
| `is_family` | Menunjukkan apakah pemesanan mencakup anak-anak atau bayi. | 
| `total_previous_bookings` | Total jumlah pemesanan sebelumnya, termasuk yang dibatalkan maupun yang tidak dibatalkan. | 
| `previous_cancellation_rate` | Proporsi pemesanan sebelumnya yang dibatalkan. | 
| `has_previous_cancellation` | Menunjukkan apakah tamu memiliki riwayat pembatalan sebelumnya. | 
| `has_booking_changes` | Menunjukkan apakah pemesanan telah diubah setelah dibuat. |
| `has_agent` | Menunjukkan apakah pemesanan menggunakan agen. | 
| `has_company` | Menunjukkan apakah pemesanan menggunakan perusahaan. |

## **5.1 Guest Composition Features**

In [39]:
df["total_guests"] = df["adults"] + df["children"] + df["babies"]

In [40]:
df["has_children"] = (df["children"] > 0).astype(int)
df["has_babies"] = (df["babies"] > 0).astype(int)
df["is_family"] = (
    (df["children"] > 0) | (df["babies"] > 0)
).astype(int)

### **Penjelasan**

Fitur `total_guests` dibuat dengan menjumlahkan jumlah orang dewasa, anak, dan bayi. Selain itu, dibuat flag `has_children`, `has_babies`, dan `is_family`.

### **Insight**

Jumlah tamu memberikan gambaran ukuran kelompok secara keseluruhan, sedangkan flag keluarga membantu model membedakan booking keluarga dan nonkeluarga. Kombinasi fitur jumlah dan flag memungkinkan model menangkap pola yang lebih sederhana maupun pola nonlinier.

## **5.2 Total Stay Nights**

In [41]:
df["total_stay_nights"] = (
    df["stays_in_weekend_nights"] +
    df["stays_in_week_nights"]
)

### **Penjelasan**

`total_stay_nights` merupakan penjumlahan malam akhir pekan dan malam hari kerja.

### **Insight**

Fitur ini memberikan gambaran durasi menginap secara utuh. Informasi tersebut lebih mudah digunakan dan diinterpretasikan daripada hanya melihat dua komponen durasi secara terpisah.

## **5.3 Booking Source Features**

In [42]:
df["has_agent"] = (df["agent"] != "no agent").astype(int)
df["has_company"] = (df["company"] != "no company").astype(int)

### **Penjelasan**

`has_agent` dan `has_company` merupakan flag biner yang menunjukkan apakah reservasi terhubung dengan agen atau perusahaan.

### **Insight**

Flag ini menyederhanakan ratusan kode identitas menjadi informasi operasional yang lebih stabil. Model dapat mempelajari perbedaan perilaku booking dengan dan tanpa perantara tanpa terlalu bergantung pada ID tertentu.

## **5.4 History Booking Features**

### **5.4.1 Total Previous Bookings**

In [43]:
df["total_previous_bookings"] = (
    df["previous_cancellations"] +
    df["previous_bookings_not_canceled"]
)

### **Penjelasan**

`total_previous_bookings` dibuat dengan menjumlahkan booking sebelumnya yang dibatalkan dan yang tidak dibatalkan.

### **Insight**

Fitur ini merepresentasikan seberapa sering seorang tamu melakukan booking di hotel ini.

### **5.4.2 Previous Cancellation Rate**

In [44]:
df["previous_cancellation_rate"] = (
    df["previous_cancellations"] /
    df["total_previous_bookings"].replace(0, np.nan)
)

### **Insight**
`previous_cancellation_rate` adalah fitur tambahan yang berisi ratio dari jumlah cancel sebelumnya dibagi total booking sebelumnya (cancel + no cancel). Hasilnya nilai antara 0 dan 1: 0 berarti tamu tersebut sebelumnya selalu datang (tidak pernah cancel), 1 berarti tamu tersebut sebelumnya selalu membatalkan booking nya, dan pecahan di antaranya menunjukkan proporsi campuran.

In [45]:
print(' === PREVIOUS_CANCELLATION_RATE MISSING VALUES ===')
print(df["previous_cancellation_rate"].isna().sum())
print(f'{df["previous_cancellation_rate"].isna().mean() * 100:.1f}% dari total baris')
print()
print('Cross-check: apakah NaN selalu bertepatan dengan total_previous_bookings == 0?')
mismatch = df[(df["previous_cancellation_rate"].isna()) & (df["total_previous_bookings"] != 0)]
print(f'Baris NaN tapi total_previous_bookings != 0 (harus 0 kalau logikanya konsisten): {len(mismatch)}')

 === PREVIOUS_CANCELLATION_RATE MISSING VALUES ===
109758
92.1% dari total baris

Cross-check: apakah NaN selalu bertepatan dengan total_previous_bookings == 0?
Baris NaN tapi total_previous_bookings != 0 (harus 0 kalau logikanya konsisten): 0


### **Insight**
 NaN pada `previous_cancellation_rate` murni berasal dari pembagian dengan `total_previous_bookings == 0` (lihat definisi fitur di 7.5), yaitu tamu yang belum pernah booking sebelumnya. Ini missing value yang *structural*, bukan acak — cross-check di atas mengonfirmasi seluruh NaN bertepatan dengan `total_previous_bookings == 0`, tidak ada baris NaN di luar kondisi itu.

In [46]:
df["previous_cancellation_rate"] = df["previous_cancellation_rate"].fillna(0)

### **Insight**
 NaN diisi 0 dengan asumsi netral: tamu tanpa riwayat booking diperlakukan seolah tidak punya riwayat pembatalan, bukan diberi rate tinggi/rendah secara sepihak. Informasi "tidak ada histori" tidak hilang karena `total_previous_bookings` tetap ada sebagai kolom terpisah — model masih bisa membedakan "rate 0 karena memang tidak pernah cancel (exposure tinggi)" vs "rate 0 karena memang belum pernah booking (exposure nol)" lewat kombinasi kedua kolom ini.

**Catatan terbuka:** `previous_cancellation_rate` sebelumnya sudah ditandai sebagai kandidat data leakage (korelasi sangat tinggi dengan `is_canceled` di tahap EDA). Mengisi NaN di sini menyelesaikan masalah kelengkapan data, **bukan** menyelesaikan masalah leakage tersebut — keputusan apakah fitur ini tetap dipakai, direkonstruksi ulang dengan batasan temporal, atau di-drop sepenuhnya perlu dibuat terpisah sebelum masuk ke tahap modeling.

### **5.4.3 Has Previous Cancellation**

In [47]:
df["has_previous_cancellation"] = (
    df["previous_cancellations"] > 0
).astype(int)

### **Insight**
`has_previous_cancellation` adalah flag biner yang menunjukkan apakah seorang tamu pernah membatalkan booking sebelumnya: bernilai 1 jika `previous_cancellations` bernilai positif (tamu memiliki riwayat membatalkan booking), dan 0 jika tidak (tamu tidak memiliki riwayat membatalkan booking).

## **5.5 Booking Modification Features**

### **5.5.1 Has Booking Changes**

In [48]:
df["has_booking_changes"] = (
    df["booking_changes"] > 0
).astype(int)

### **Insight**
`has_booking_changes` adalah flag biner yang menunjukkan apakah seorang tamu pernah melakukan perubahan pada booking sebelumnya: bernilai 1 jika `booking_changes` bernilai positif (tamu memiliki riwayat melakukan perubahan pada booking sebelumnya), dan 0 jika tidak (tamu tidak memiliki riwayat melakukan perubahan pada booking sebelumnya).

### **5.5.2 Room Type Changed**

In [49]:
df["room_type_changed"] = (
    df["reserved_room_type"] != df["assigned_room_type"]
).astype(int)

### **Insight**
`room_type_changed` merupakan flag biner: 1 kalau kamar yang diberikan (`assigned_room_type`) BERBEDA dengan kamar yang awalnya dipesan (`reserved_room_type`), dan 0 kalau kamar yang diberikan SAMA dengan kamar yang awalnya dipesan.

# **Section 6: Post-Cleaning Validation**

In [50]:
print("Dataset shape before cleaning:", df_raw.shape)
print("Final dataset shape:", df.shape)
print("Total missing values:", df.isna().sum().sum())
print("Total duplicated rows:", df.duplicated().sum())
print("Zero-guest records:",
      ((df["adults"] + df["children"] + df["babies"]) == 0).sum())
print("Negative ADR records:", (df["adr"] < 0).sum())

display(df["is_canceled"].value_counts())
display(df["is_canceled"].value_counts(normalize=True).mul(100).round(2))

Dataset shape before cleaning: (119390, 32)
Final dataset shape: (119205, 42)
Total missing values: 0
Total duplicated rows: 32238
Zero-guest records: 0
Negative ADR records: 0


is_canceled
0    75008
1    44197
Name: count, dtype: int64

is_canceled
0    62.92
1    37.08
Name: proportion, dtype: float64

### **Penjelasan**

Post-cleaning validation dilakukan untuk memastikan bahwa seluruh proses pembersihan dan feature engineering telah menghasilkan dataset yang konsisten dan siap diproses pada tahap selanjutnya.

Dataset akhir terdiri dari **119.205 baris dan 42 kolom**. Dibandingkan dengan 119.390 baris pada data awal, sebanyak **185 catatan telah dihapus** berdasarkan aturan validasi yang telah ditetapkan.

### **Temuan**

- Seluruh missing value telah ditangani sehingga tidak terdapat nilai kosong pada dataset akhir.
- Tidak ditemukan lagi reservasi dengan total tamu nol.
- Tidak ditemukan lagi nilai ADR negatif.
- Cancellation rate setelah cleaning adalah **37,08%**, relatif stabil dibandingkan kondisi awal sebesar **37,04%**.
- Masih terdapat **32.238 baris identik** berdasarkan seluruh kolom yang tersedia.

### **Insight**

Proses cleaning berhasil memperbaiki ketidakkonsistenan utama tanpa mengubah distribusi target secara material. Cancellation rate hanya berubah sebesar **0,04 poin persentase**, sehingga penghapusan data tidak menyebabkan perubahan berarti pada representasi kelas pembatalan.

Tidak adanya missing value, reservasi tanpa tamu, dan ADR negatif menunjukkan bahwa aturan cleaning telah diterapkan secara konsisten.

Meskipun demikian, keberadaan 32.238 baris identik tetap perlu didokumentasikan. Karena dataset tidak menyediakan ID reservasi unik, baris yang identik belum dapat dipastikan sebagai penggandaan transaksi yang sama dan tidak sebaiknya langsung dihapus tanpa validasi tambahan.

### **Validation Decision**

Dataset telah memenuhi pemeriksaan kelengkapan dan validitas dasar. Dataset dapat dilanjutkan ke tahap exploratory data analysis dan modelling setelah:

1. Memastikan `reservation_status` dan `reservation_status_date` sudah dikeluarkan dari fitur model.
2. Mendokumentasikan keputusan mengenai baris identik.
3. Memastikan seluruh fitur hanya menggunakan informasi yang tersedia pada waktu prediksi.

### 7. Export

In [51]:
df.to_csv("../data/hotel_booking_2017_cleaned.csv", index=False)